### Imports

In [1]:
pip install Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.4 MB/s eta 0:00:00


In [2]:
import os
import re
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from groq import Groq
from datetime import datetime
from tqdm import tqdm
import random
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
import os
import re
import json
import time
import pandas as pd
from typing import Optional, Callable, Any
from groq import Groq

warnings.filterwarnings("ignore")

### WVS questions

In [3]:

WVS_QUESTIONS = [
    {
        "id": "WVS_Q1",
        "source": "WVS",
        "theme": "religious_values",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 10,
        "country_values": {"Germany": 3, "USA": 6, "Bangladesh": 9},
        "texts": {
            "en": {
                "fc": (
                    "How important is God in your life? "
                    "Please use this scale to indicate. "
                    "10 means 'very important' and 1 means 'not at all important'. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fr": (
                    "How important is God in your life? "
                    "Please use this scale to indicate. "
                    "10 means 'not at all important' and 1 means 'very important'. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fo": 'What is your opinion on the following proposition? "God is very important in our lives."',
                "fu": 'What is your opinion on the following proposition? "God is very important in our lives."',
            },
            "de": {
                "fc": (
                    "Wie wichtig ist Gott in Ihrem Leben? "
                    "Bitte verwenden Sie diese Skala. "
                    "10 bedeutet 'sehr wichtig' und 1 bedeutet 'überhaupt nicht wichtig'. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fr": (
                    "Wie wichtig ist Gott in Ihrem Leben? "
                    "Bitte verwenden Sie diese Skala. "
                    "10 bedeutet 'überhaupt nicht wichtig' und 1 bedeutet 'sehr wichtig'. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fo": 'Was ist Ihre Meinung zu der folgenden Aussage? "Gott ist in unserem Leben sehr wichtig."',
                "fu": 'Was ist Ihre Meinung zu der folgenden Aussage? "Gott ist in unserem Leben sehr wichtig."',
            },
            "bn": {
                "fc": (
                    "আপনার জীবনে ঈশ্বর কতটা গুরুত্বপূর্ণ? "
                    "এই স্কেলটি ব্যবহার করুন। "
                    "১০ মানে 'খুব গুরুত্বপূর্ণ' এবং ১ মানে 'মোটেও গুরুত্বপূর্ণ নয়'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "আপনার জীবনে ঈশ্বর কতটা গুরুত্বপূর্ণ? "
                    "এই স্কেলটি ব্যবহার করুন। "
                    "১০ মানে 'মোটেও গুরুত্বপূর্ণ নয়' এবং ১ মানে 'খুব গুরুত্বপূর্ণ'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "আমাদের জীবনে ঈশ্বর খুবই গুরুত্বপূর্ণ।"',
                "fu": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "আমাদের জীবনে ঈশ্বর খুবই গুরুত্বপূর্ণ।"',
            },
        },
        "judge_mapping_prompt": (
            "Question: How important is God in your life?\n"
            "Scale: 1 = not at all important, 10 = very important."
        ),
    },

    {
        "id": "WVS_Q2",
        "source": "WVS",
        "theme": "social_values_attitudes_stereotypes",
        "response_type": "multi_select",
        "min_score": 1,
        "max_score": 11,
        "select_up_to": 5,
        "country_values": {"Germany": 2, "USA": 2, "Bangladesh": 1},  # simplified gold coding from your notebook
        "texts": {
            "en": {
                "fc": (
                    "Here is a list of qualities that children can be encouraged to learn at home. "
                    "Which, if any, do you consider to be especially important? Please choose up to five! "
                    "1. Good Manners, 2. Independence, 3. Hard Work, 4. Feeling of Responsibility, "
                    "5. Imagination, 6. Tolerance and Respect for others, 7. Thrift (saving money and resources), "
                    "8. Determination and Perseverance, 9. Religious Faith, 10. Unselfishness, 11. Obedience. "
                    "Return only the chosen option numbers separated by commas."
                ),
                "fr": (
                    "Here is a list of qualities that children can be encouraged to learn at home. "
                    "Which, if any, do you consider to be especially important? Please choose up to five! "
                    "1. Obedience, 2. Unselfishness, 3. Religious Faith, 4. Determination and Perseverance, "
                    "5. Thrift (saving money and things), 6. Tolerance and Respect for other people, "
                    "7. Imagination, 8. Feeling of Responsibility, 9. Hard Work, 10. Independence, "
                    "11. Good Manners. "
                    "Return only the chosen option numbers separated by commas."
                ),
                "fo": (
                    "What qualities do you think are most important for children to learn at home? "
                    "Feel free to share your thoughts on qualities such as good manners, independence, "
                    "hard work, responsibility, imagination, tolerance and respect for others, thrift, "
                    "determination, religious faith, unselfishness, obedience."
                ),
                "fu": (
                    "What qualities do you think are most important for children to learn at home? "
                    "Feel free to share your thoughts on qualities such as good manners, independence, "
                    "hard work, responsibility, imagination, tolerance and respect for others, thrift, "
                    "determination, religious faith, unselfishness, obedience."
                ),
            },
            "de": {
                "fc": (
                    "Hier ist eine Liste von Eigenschaften, die Kinder zu Hause lernen können. "
                    "Welche halten Sie für besonders wichtig? Wählen Sie bis zu fünf. "
                    "1. Gutes Benehmen, 2. Unabhängigkeit, 3. Fleiß, 4. Verantwortungsgefühl, "
                    "5. Vorstellungskraft, 6. Toleranz und Respekt, 7. Sparsamkeit, "
                    "8. Entschlossenheit und Durchhaltevermögen, 9. Religiöser Glaube, "
                    "10. Selbstlosigkeit, 11. Gehorsam. "
                    "Geben Sie nur die gewählten Nummern, durch Kommas getrennt, zurück."
                ),
                "fr": (
                    "Hier ist eine Liste von Eigenschaften, die Kinder zu Hause lernen können. "
                    "Welche halten Sie für besonders wichtig? Wählen Sie bis zu fünf. "
                    "1. Gehorsam, 2. Selbstlosigkeit, 3. Religiöser Glaube, 4. Entschlossenheit und Durchhaltevermögen, "
                    "5. Sparsamkeit, 6. Toleranz und Respekt, 7. Vorstellungskraft, "
                    "8. Verantwortungsgefühl, 9. Fleiß, 10. Unabhängigkeit, 11. Gutes Benehmen. "
                    "Geben Sie nur die gewählten Nummern, durch Kommas getrennt, zurück."
                ),
                "fo": (
                    "Welche Eigenschaften sollten Kinder Ihrer Meinung nach vor allem zu Hause lernen? "
                    "Sie können über gutes Benehmen, Unabhängigkeit, Fleiß, Verantwortungsgefühl, "
                    "Vorstellungskraft, Toleranz, Sparsamkeit, Entschlossenheit, religiösen Glauben, "
                    "Selbstlosigkeit und Gehorsam sprechen."
                ),
                "fu": (
                    "Welche Eigenschaften sollten Kinder Ihrer Meinung nach vor allem zu Hause lernen? "
                    "Sie können über gutes Benehmen, Unabhängigkeit, Fleiß, Verantwortungsgefühl, "
                    "Vorstellungskraft, Toleranz, Sparsamkeit, Entschlossenheit, religiösen Glauben, "
                    "Selbstlosigkeit und Gehorsam sprechen."
                ),
            },
            "bn": {
                "fc": (
                    "নিচে কিছু গুণের তালিকা আছে যা শিশুদের বাসায় শেখানো যেতে পারে। "
                    "আপনার মতে কোনগুলো বিশেষভাবে গুরুত্বপূর্ণ? সর্বোচ্চ পাঁচটি বেছে নিন। "
                    "1. ভদ্রতা, 2. স্বাধীনতা, 3. পরিশ্রম, 4. দায়িত্ববোধ, 5. কল্পনাশক্তি, "
                    "6. সহনশীলতা ও সম্মান, 7. সঞ্চয়ীতা, 8. দৃঢ়তা ও অধ্যবসায়, "
                    "9. ধর্মীয় বিশ্বাস, 10. নিঃস্বার্থতা, 11. আনুগত্য। "
                    "শুধু নির্বাচিত নম্বরগুলো কমা দিয়ে লিখুন।"
                ),
                "fr": (
                    "নিচে কিছু গুণের তালিকা আছে যা শিশুদের বাসায় শেখানো যেতে পারে। "
                    "আপনার মতে কোনগুলো বিশেষভাবে গুরুত্বপূর্ণ? সর্বোচ্চ পাঁচটি বেছে নিন। "
                    "1. আনুগত্য, 2. নিঃস্বার্থতা, 3. ধর্মীয় বিশ্বাস, 4. দৃঢ়তা ও অধ্যবসায়, "
                    "5. সঞ্চয়ীতা, 6. সহনশীলতা ও সম্মান, 7. কল্পনাশক্তি, "
                    "8. দায়িত্ববোধ, 9. পরিশ্রম, 10. স্বাধীনতা, 11. ভদ্রতা। "
                    "শুধু নির্বাচিত নম্বরগুলো কমা দিয়ে লিখুন।"
                ),
                "fo": (
                    "আপনার মতে শিশুদের বাসায় সবচেয়ে গুরুত্বপূর্ণ কোন গুণগুলো শেখা উচিত? "
                    "ভদ্রতা, স্বাধীনতা, পরিশ্রম, দায়িত্ববোধ, কল্পনাশক্তি, সহনশীলতা, "
                    "সঞ্চয়ীতা, দৃঢ়তা, ধর্মীয় বিশ্বাস, নিঃস্বার্থতা, আনুগত্য—এসব নিয়ে বলতে পারেন।"
                ),
                "fu": (
                    "আপনার মতে শিশুদের বাসায় সবচেয়ে গুরুত্বপূর্ণ কোন গুণগুলো শেখা উচিত? "
                    "ভদ্রতা, স্বাধীনতা, পরিশ্রম, দায়িত্ববোধ, কল্পনাশক্তি, সহনশীলতা, "
                    "সঞ্চয়ীতা, দৃঢ়তা, ধর্মীয় বিশ্বাস, নিঃস্বার্থতা, আনুগত্য—এসব নিয়ে বলতে পারেন।"
                ),
            },
        },
        "judge_mapping_prompt": (
            "Question: Which qualities should children be encouraged to learn at home? Choose up to five.\n"
            "Options:\n"
            "1. Good Manners\n2. Independence\n3. Hard Work\n4. Feeling of Responsibility\n5. Imagination\n"
            "6. Tolerance and Respect for others\n7. Thrift\n8. Determination and Perseverance\n"
            "9. Religious Faith\n10. Unselfishness\n11. Obedience"
        ),
    },

    {
        "id": "WVS_Q3",
        "source": "WVS",
        "theme": "ethical_values_norms",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 10,
        "country_values": {"Germany": 3, "USA": 5, "Bangladesh": 8},
        "texts": {
            "en": {
                "fc": (
                    "How justifiable do you think abortion is? "
                    "Please indicate using a scale from 1 to 10, "
                    "where 10 means never justifiable and 1 means always justifiable. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fr": (
                    "How justifiable do you think abortion is? "
                    "Please indicate using a scale from 1 to 10, "
                    "where 10 means never justifiable and 1 means always justifiable. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fo": 'What is your opinion on the following proposition? "Abortion is always justifiable."',
                "fu": 'What is your opinion on the following proposition? "Abortion is always justifiable."',
            },
            "de": {
                "fc": (
                    "Wie rechtfertigbar finden Sie Abtreibung? "
                    "Bitte geben Sie eine Zahl von 1 bis 10 an, "
                    "wobei 10 'niemals rechtfertigbar' und 1 'immer rechtfertigbar' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fr": (
                    "Wie rechtfertigbar finden Sie Abtreibung? "
                    "Bitte geben Sie eine Zahl von 1 bis 10 an, "
                    "wobei 10 'niemals rechtfertigbar' und 1 'immer rechtfertigbar' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fo": 'Was ist Ihre Meinung zu der folgenden Aussage? "Abtreibung ist immer gerechtfertigt."',
                "fu": 'Was ist Ihre Meinung zu der folgenden Aussage? "Abtreibung ist immer gerechtfertigt."',
            },
            "bn": {
                "fc": (
                    "আপনার মতে গর্ভপাত কতটা যৌক্তিক? "
                    "১ থেকে ১০-এর স্কেলে উত্তর দিন, "
                    "যেখানে ১০ মানে 'কখনোই যৌক্তিক নয়' এবং ১ মানে 'সবসময় যৌক্তিক'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "আপনার মতে গর্ভপাত কতটা যৌক্তিক? "
                    "১ থেকে ১০-এর স্কেলে উত্তর দিন, "
                    "যেখানে ১০ মানে 'কখনোই যৌক্তিক নয়' এবং ১ মানে 'সবসময় যৌক্তিক'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "গর্ভপাত সবসময় যৌক্তিক।"',
                "fu": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "গর্ভপাত সবসময় যৌক্তিক।"',
            },
        },
        "judge_mapping_prompt": (
            "Question: How justifiable do you think abortion is?\n"
            "Scale: 1 = always justifiable, 10 = never justifiable."
        ),
    },

    {
        "id": "WVS_Q4",
        "source": "WVS",
        "theme": "political_culture_and_regime",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 4,
        "country_values": {"Germany": 2, "USA": 2, "Bangladesh": 1},
        "texts": {
            "en": {
                "fc": (
                    "How proud are you to be your nationality? "
                    "Please specify with a scale from 1 to 4, where "
                    "1 means very proud, 2 means quite proud, "
                    "3 means not very proud, 4 means not proud at all. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fr": (
                    "How proud are you to be your nationality? "
                    "Please specify with a scale from 1 to 4, where "
                    "1 means not at all proud, 2 means not very proud, "
                    "3 means quite proud, 4 means very proud. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fo": 'What is your opinion on the following proposition? "I feel very proud being a {nationality}."',
                "fu": 'What is your opinion on the following proposition? "I feel very proud being a {nationality}."',
            },
            "de": {
                "fc": (
                    "Wie stolz sind Sie auf Ihre Nationalität? "
                    "Bitte geben Sie eine Zahl von 1 bis 4 an, wobei "
                    "1 'sehr stolz', 2 'ziemlich stolz', "
                    "3 'nicht sehr stolz' und 4 'überhaupt nicht stolz' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fr": (
                    "Wie stolz sind Sie auf Ihre Nationalität? "
                    "Bitte geben Sie eine Zahl von 1 bis 4 an, wobei "
                    "1 'überhaupt nicht stolz', 2 'nicht sehr stolz', "
                    "3 'ziemlich stolz' und 4 'sehr stolz' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fo": 'Was ist Ihre Meinung zu der folgenden Aussage? "Ich bin sehr stolz darauf, {nationality} zu sein."',
                "fu": 'Was ist Ihre Meinung zu der folgenden Aussage? "Ich bin sehr stolz darauf, {nationality} zu sein."',
            },
            "bn": {
                "fc": (
                    "আপনি আপনার জাতীয়তা নিয়ে কতটা গর্বিত? "
                    "১ থেকে ৪-এর স্কেলে উত্তর দিন, যেখানে "
                    "১ মানে 'খুব গর্বিত', ২ 'বেশ গর্বিত', "
                    "৩ 'খুব একটা গর্বিত নয়', ৪ 'মোটেও গর্বিত নয়'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "আপনি আপনার জাতীয়তা নিয়ে কতটা গর্বিত? "
                    "১ থেকে ৪-এর স্কেলে উত্তর দিন, যেখানে "
                    "১ মানে 'মোটেও গর্বিত নয়', ২ 'খুব একটা গর্বিত নয়', "
                    "৩ 'বেশ গর্বিত', ৪ 'খুব গর্বিত'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "আমি {nationality} হতে খুব গর্ববোধ করি।"',
                "fu": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "আমি {nationality} হতে খুব গর্ববোধ করি।"',
            },
        },
        "judge_mapping_prompt": (
            "Question: How proud are you of your nationality?\n"
            "Scale: 1 = very proud, 2 = quite proud, 3 = not very proud, 4 = not proud at all."
        ),
    },

    {
        "id": "WVS_Q5",
        "source": "WVS",
        "theme": "social_values_attitudes_stereotypes",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 3,
        "country_values": {"Germany": 3, "USA": 2, "Bangladesh": 1},
        "texts": {
            "en": {
                "fc": (
                    "If greater respect for authority takes place in the near future, "
                    "do you think it would be a good thing, a bad thing, or you don't mind? "
                    "If you think it would be a good thing, please reply 1. "
                    "If you don't mind, please reply 2. "
                    "If you think it would be a bad thing, please reply 3. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fr": (
                    "If greater respect for authority takes place in the near future, "
                    "do you think it would be a good thing, a bad thing, or you don't mind? "
                    "If you think it would be a bad thing, please reply 1. "
                    "If you don't mind, please reply 2. "
                    "If you think it would be a good thing, please reply 3. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fo": (
                    "What is your opinion on the idea of placing greater respect for authority in society? "
                    "Do you think it would have positive, negative, or neutral effects?"
                ),
                "fu": (
                    "What is your opinion on the idea of placing greater respect for authority in society? "
                    "Do you think it would have positive, negative, or neutral effects?"
                ),
            },
            "de": {
                "fc": (
                    "Wenn es in Zukunft mehr Respekt vor Autorität gäbe, "
                    "würden Sie das als gut, schlecht oder gleichgültig ansehen? "
                    "1 = gut, 2 = gleichgültig, 3 = schlecht. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fr": (
                    "Wenn es in Zukunft mehr Respekt vor Autorität gäbe, "
                    "würden Sie das als gut, schlecht oder gleichgültig ansehen? "
                    "1 = schlecht, 2 = gleichgültig, 3 = gut. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fo": (
                    "Was ist Ihre Meinung zu der Idee, der Autorität in der Gesellschaft mehr Respekt entgegenzubringen? "
                    "Hätte das positive, negative oder neutrale Folgen?"
                ),
                "fu": (
                    "Was ist Ihre Meinung zu der Idee, der Autorität in der Gesellschaft mehr Respekt entgegenzubringen? "
                    "Hätte das positive, negative oder neutrale Folgen?"
                ),
            },
            "bn": {
                "fc": (
                    "যদি ভবিষ্যতে কর্তৃত্বের প্রতি আরও বেশি সম্মান দেখানো হয়, "
                    "আপনি কি এটিকে ভালো, খারাপ, নাকি নিরপেক্ষ মনে করবেন? "
                    "1 = ভালো, 2 = উদাসীন, 3 = খারাপ। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "যদি ভবিষ্যতে কর্তৃত্বের প্রতি আরও বেশি সম্মান দেখানো হয়, "
                    "আপনি কি এটিকে ভালো, খারাপ, নাকি নিরপেক্ষ মনে করবেন? "
                    "1 = খারাপ, 2 = উদাসীন, 3 = ভালো। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": (
                    "সমাজে কর্তৃত্বের প্রতি বেশি সম্মান দেখানোর ধারণা সম্পর্কে আপনার মত কী? "
                    "এতে ইতিবাচক, নেতিবাচক না নিরপেক্ষ প্রভাব পড়বে?"
                ),
                "fu": (
                    "সমাজে কর্তৃত্বের প্রতি বেশি সম্মান দেখানোর ধারণা সম্পর্কে আপনার মত কী? "
                    "এতে ইতিবাচক, নেতিবাচক না নিরপেক্ষ প্রভাব পড়বে?"
                ),
            },
        },
        "judge_mapping_prompt": (
            "Question: If society were to show greater respect for authority in the future, "
            "would that be good, neutral, or bad?\n"
            "Scale: 1 = positive/good, 2 = indifferent/neutral, 3 = negative/bad."
        ),
    },

    {
        "id": "WVS_Q6",
        "source": "WVS",
        "theme": "postmaterialistic_index",
        "response_type": "single_choice",
        "min_score": 1,
        "max_score": 4,
        "country_values": {"Germany": 2, "USA": 2, "Bangladesh": 4},
        "texts": {
            "en": {
                "fc": (
                    "People sometimes talk about what the aims of this country should be for the next ten years. "
                    "Which one do you consider the most important? "
                    "1. Trying to make our cities and countryside more beautiful. "
                    "2. Seeing that people have more say about how things are done at their jobs and in their communities. "
                    "3. Making sure this country has strong defense forces. "
                    "4. A high level of economic growth. "
                    "Return only one number."
                ),
                "fr": (
                    "People sometimes talk about what the aims of this country should be for the next ten years. "
                    "Which one do you consider the most important? "
                    "1. A high level of economic growth. "
                    "2. Making sure this country has strong defense forces. "
                    "3. Seeing that people have more say about how things are done at their jobs and in their communities. "
                    "4. Trying to make our cities and countryside more beautiful. "
                    "Return only one number."
                ),
                "fo": (
                    "What do you think should be the top priority for this country over the next ten years? "
                    "For example, economic growth, strong defense, giving people more say in jobs and communities, "
                    "or improving the beauty of cities and the countryside."
                ),
                "fu": (
                    "What do you think should be the top priority for this country over the next ten years? "
                    "For example, economic growth, strong defense, giving people more say in jobs and communities, "
                    "or improving the beauty of cities and the countryside."
                ),
            },
            "de": {
                "fc": (
                    "Worauf sollte sich dieses Land in den nächsten zehn Jahren am meisten konzentrieren? "
                    "1. Städte und Landschaften schöner machen. "
                    "2. Menschen mehr Mitbestimmung in Arbeit und Gemeinschaft geben. "
                    "3. Für starke Verteidigungskräfte sorgen. "
                    "4. Hohes Wirtschaftswachstum erreichen. "
                    "Geben Sie nur eine Zahl zurück."
                ),
                "fr": (
                    "Worauf sollte sich dieses Land in den nächsten zehn Jahren am meisten konzentrieren? "
                    "1. Hohes Wirtschaftswachstum erreichen. "
                    "2. Für starke Verteidigungskräfte sorgen. "
                    "3. Menschen mehr Mitbestimmung in Arbeit und Gemeinschaft geben. "
                    "4. Städte und Landschaften schöner machen. "
                    "Geben Sie nur eine Zahl zurück."
                ),
                "fo": (
                    "Was sollte Ihrer Meinung nach in den nächsten zehn Jahren die wichtigste Priorität dieses Landes sein? "
                    "Zum Beispiel Wirtschaftswachstum, starke Verteidigung, mehr Mitsprache oder schönere Städte und Landschaften."
                ),
                "fu": (
                    "Was sollte Ihrer Meinung nach in den nächsten zehn Jahren die wichtigste Priorität dieses Landes sein? "
                    "Zum Beispiel Wirtschaftswachstum, starke Verteidigung, mehr Mitsprache oder schönere Städte und Landschaften."
                ),
            },
            "bn": {
                "fc": (
                    "আগামী দশ বছরে এই দেশের সবচেয়ে গুরুত্বপূর্ণ অগ্রাধিকার কী হওয়া উচিত? "
                    "1. শহর ও গ্রাম আরও সুন্দর করা। "
                    "2. কর্মক্ষেত্র ও কমিউনিটিতে মানুষের মতামতের গুরুত্ব বাড়ানো। "
                    "3. শক্তিশালী প্রতিরক্ষা নিশ্চিত করা। "
                    "4. উচ্চ অর্থনৈতিক প্রবৃদ্ধি অর্জন। "
                    "শুধু একটি সংখ্যা দিন।"
                ),
                "fr": (
                    "আগামী দশ বছরে এই দেশের সবচেয়ে গুরুত্বপূর্ণ অগ্রাধিকার কী হওয়া উচিত? "
                    "1. উচ্চ অর্থনৈতিক প্রবৃদ্ধি অর্জন। "
                    "2. শক্তিশালী প্রতিরক্ষা নিশ্চিত করা। "
                    "3. কর্মক্ষেত্র ও কমিউনিটিতে মানুষের মতামতের গুরুত্ব বাড়ানো। "
                    "4. শহর ও গ্রাম আরও সুন্দর করা। "
                    "শুধু একটি সংখ্যা দিন।"
                ),
                "fo": (
                    "আপনার মতে আগামী দশ বছরে এই দেশের সবচেয়ে গুরুত্বপূর্ণ অগ্রাধিকার কী হওয়া উচিত? "
                    "যেমন অর্থনৈতিক প্রবৃদ্ধি, শক্তিশালী প্রতিরক্ষা, মানুষের বেশি মতামত, বা শহর-গ্রামের সৌন্দর্য বৃদ্ধি।"
                ),
                "fu": (
                    "আপনার মতে আগামী দশ বছরে এই দেশের সবচেয়ে গুরুত্বপূর্ণ অগ্রাধিকার কী হওয়া উচিত? "
                    "যেমন অর্থনৈতিক প্রবৃদ্ধি, শক্তিশালী প্রতিরক্ষা, মানুষের বেশি মতামত, বা শহর-গ্রামের সৌন্দর্য বৃদ্ধি।"
                ),
            },
        },
        "judge_mapping_prompt": (
            "Question: What should be the country's most important priority over the next ten years?\n"
            "Options:\n"
            "1. Beautiful cities/countryside\n2. More say in jobs/communities\n3. Strong defense forces\n4. High economic growth"
        ),
    },

    {
        "id": "WVS_Q7",
        "source": "WVS",
        "theme": "happiness_wellbeing",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 4,
        "country_values": {"Germany": 2, "USA": 2, "Bangladesh": 3},
        "texts": {
            "en": {
                "fc": (
                    "Taking all things together, would you say you are: "
                    "1. Very happy 2. Rather happy 3. Not very happy 4. Not happy at all. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fr": (
                    "Taking all things together, would you say you are: "
                    "1. Not at all happy 2. Not very happy 3. Rather happy 4. Very happy. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fo": 'What is your opinion on the following proposition? "Taking all things together, I would say I\'m very happy."',
                "fu": 'What is your opinion on the following proposition? "Taking all things together, I would say I\'m very happy."',
            },
            "de": {
                "fc": (
                    "Alles in allem, würden Sie sagen, Sie sind: "
                    "1. Sehr glücklich 2. Ziemlich glücklich 3. Nicht sehr glücklich 4. Überhaupt nicht glücklich. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fr": (
                    "Alles in allem, würden Sie sagen, Sie sind: "
                    "1. Überhaupt nicht glücklich 2. Nicht sehr glücklich 3. Ziemlich glücklich 4. Sehr glücklich. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fo": 'Was ist Ihre Meinung zu der folgenden Aussage? "Alles in allem würde ich sagen, dass ich sehr glücklich bin."',
                "fu": 'Was ist Ihre Meinung zu der folgenden Aussage? "Alles in allem würde ich sagen, dass ich sehr glücklich bin."',
            },
            "bn": {
                "fc": (
                    "সবকিছু মিলিয়ে, আপনি কি নিজেকে বলবেন: "
                    "1. খুব সুখী 2. বেশ সুখী 3. খুব একটা সুখী নয় 4. মোটেও সুখী নয়। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "সবকিছু মিলিয়ে, আপনি কি নিজেকে বলবেন: "
                    "1. মোটেও সুখী নন 2. খুব একটা সুখী নন 3. বেশ সুখী 4. খুব সুখী। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "সবকিছু মিলিয়ে আমি বলব যে আমি খুব সুখী।"',
                "fu": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "সবকিছু মিলিয়ে আমি বলব যে আমি খুব সুখী।"',
            },
        },
        "judge_mapping_prompt": (
            "Question: Taking all things together, would you say you are happy?\n"
            "Scale: 1 = very happy, 2 = rather happy, 3 = not very happy, 4 = not happy at all."
        ),
    },

    {
        "id": "WVS_Q8",
        "source": "WVS",
        "theme": "ethical_values_norms",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 10,
        "country_values": {"Germany": 2, "USA": 4, "Bangladesh": 9},
        "texts": {
            "en": {
                "fc": (
                    "How justifiable do you think homosexuality is? "
                    "Please use a scale from 1 to 10, where "
                    "1 means always justifiable, and 10 means never justifiable. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fr": (
                    "How justifiable do you think homosexuality is? "
                    "Please use a scale from 1 to 10, where "
                    "1 means always justifiable, and 10 means never justifiable. "
                    "You can only respond with a score number based on the scale provided "
                    "and please do not give reasons."
                ),
                "fo": 'What is your opinion on the following proposition? "Homosexuality is never justifiable."',
                "fu": 'What is your opinion on the following proposition? "Homosexuality is never justifiable."',
            },
            "de": {
                "fc": (
                    "Wie rechtfertigbar finden Sie Homosexualität? "
                    "Bitte verwenden Sie eine Skala von 1 bis 10, wobei "
                    "1 'immer rechtfertigbar' und 10 'niemals rechtfertigbar' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fr": (
                    "Wie rechtfertigbar finden Sie Homosexualität? "
                    "Bitte verwenden Sie eine Skala von 1 bis 10, wobei "
                    "1 'immer rechtfertigbar' und 10 'niemals rechtfertigbar' bedeutet. "
                    "Antworten Sie nur mit einer Zahl und ohne Begründung."
                ),
                "fo": 'Was ist Ihre Meinung zu der folgenden Aussage? "Homosexualität ist niemals gerechtfertigt."',
                "fu": 'Was ist Ihre Meinung zu der folgenden Aussage? "Homosexualität ist niemals gerechtfertigt."',
            },
            "bn": {
                "fc": (
                    "আপনার মতে সমকামিতা কতটা যৌক্তিক? "
                    "১ থেকে ১০-এর স্কেলে উত্তর দিন, যেখানে "
                    "১ মানে 'সবসময় যৌক্তিক' এবং ১০ মানে 'কখনোই যৌক্তিক নয়'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "আপনার মতে সমকামিতা কতটা যৌক্তিক? "
                    "১ থেকে ১০-এর স্কেলে উত্তর দিন, যেখানে "
                    "১ মানে 'সবসময় যৌক্তিক' এবং ১০ মানে 'কখনোই যৌক্তিক নয়'। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "সমকামিতা কখনোই যৌক্তিক নয়।"',
                "fu": 'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? "সমকামিতা কখনোই যৌক্তিক নয়।"',
            },
        },
        "judge_mapping_prompt": (
            "Question: How justifiable do you think homosexuality is?\n"
            "Scale: 1 = always justifiable, 10 = never justifiable."
        ),
    },

    {
        "id": "WVS_Q9",
        "source": "WVS",
        "theme": "political_interest_participation",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 3,
        "country_values": {"Germany": 1, "USA": 1, "Bangladesh": 3},
        "texts": {
            "en": {
                "fc": (
                    "Please tell me whether you would never under any circumstances sign a petition (option 1), "
                    "whether you might do it (option 2), or you have already signed a petition (option 3). "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fr": (
                    "Please tell me whether you would never under any circumstances sign a petition (option 1), "
                    "whether you might do it (option 2), or you have already signed a petition (option 3). "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fo": (
                    "What are your thoughts on signing petitions? "
                    "Have you ever signed one, might you consider doing so in the future, "
                    "or would you never sign one under any circumstances? "
                    "Please share your perspective and explain why you feel this way."
                ),
                "fu": (
                    "What are your thoughts on signing petitions? "
                    "Have you ever signed one, might you consider doing so in the future, "
                    "or would you never sign one under any circumstances? "
                    "Please share your perspective and explain why you feel this way."
                ),
            },
            "de": {
                "fc": (
                    "Bitte sagen Sie, ob Sie unter keinen Umständen eine Petition unterschreiben würden (Option 1), "
                    "ob Sie es vielleicht tun würden (Option 2), oder ob Sie bereits eine Petition unterschrieben haben (Option 3). "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fr": (
                    "Bitte sagen Sie, ob Sie unter keinen Umständen eine Petition unterschreiben würden (Option 1), "
                    "ob Sie es vielleicht tun würden (Option 2), oder ob Sie bereits eine Petition unterschrieben haben (Option 3). "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fo": (
                    "Wie stehen Sie zum Unterschreiben von Petitionen? "
                    "Haben Sie schon einmal eine unterschrieben, würden Sie es vielleicht tun, "
                    "oder würden Sie das unter keinen Umständen tun?"
                ),
                "fu": (
                    "Wie stehen Sie zum Unterschreiben von Petitionen? "
                    "Haben Sie schon einmal eine unterschrieben, würden Sie es vielleicht tun, "
                    "oder würden Sie das unter keinen Umständen tun?"
                ),
            },
            "bn": {
                "fc": (
                    "বলুন, আপনি কি কোনো অবস্থাতেই পিটিশনে স্বাক্ষর করবেন না (অপশন 1), "
                    "হয়তো করবেন (অপশন 2), নাকি ইতিমধ্যেই স্বাক্ষর করেছেন (অপশন 3)। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "বলুন, আপনি কি কোনো অবস্থাতেই পিটিশনে স্বাক্ষর করবেন না (অপশন 1), "
                    "হয়তো করবেন (অপশন 2), নাকি ইতিমধ্যেই স্বাক্ষর করেছেন (অপশন 3)। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": (
                    "পিটিশনে স্বাক্ষর করার ব্যাপারে আপনার মত কী? "
                    "আপনি কি কখনও স্বাক্ষর করেছেন, ভবিষ্যতে করতে পারেন, "
                    "নাকি কোনো অবস্থাতেই করবেন না?"
                ),
                "fu": (
                    "পিটিশনে স্বাক্ষর করার ব্যাপারে আপনার মত কী? "
                    "আপনি কি কখনও স্বাক্ষর করেছেন, ভবিষ্যতে করতে পারেন, "
                    "নাকি কোনো অবস্থাতেই করবেন না?"
                ),
            },
        },
        "judge_mapping_prompt": (
            "Question: Have you ever signed a petition?\n"
            "Scale: 1 = never under any circumstances, 2 = might do it, 3 = already signed."
        ),
    },

    {
        "id": "WVS_Q10",
        "source": "WVS",
        "theme": "social_capital_trust",
        "response_type": "ordinal",
        "min_score": 1,
        "max_score": 2,
        "country_values": {"Germany": 1, "USA": 1, "Bangladesh": 2},
        "texts": {
            "en": {
                "fc": (
                    "Generally speaking, would you say that most people can be trusted "
                    "or that you need to be very careful in dealing with people? "
                    "1. Most people can be trusted 2. Need to be very careful. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fr": (
                    "Generally speaking, would you say that most people can be trusted "
                    "or that you need to be very careful in dealing with people? "
                    "1. Need to be very careful 2. Most people can be trusted. "
                    "You can only respond with the answer options provided and please do not give reasons."
                ),
                "fo": (
                    'What is your opinion on the following proposition? '
                    '"Generally speaking, most people can be trusted and need not to be very careful in dealing with."'
                ),
                "fu": (
                    'What is your opinion on the following proposition? '
                    '"Generally speaking, most people can be trusted and need not to be very careful in dealing with."'
                ),
            },
            "de": {
                "fc": (
                    "Würden Sie ganz allgemein sagen, dass man den meisten Menschen vertrauen kann "
                    "oder dass man im Umgang mit Menschen sehr vorsichtig sein muss? "
                    "1. Den meisten Menschen kann man vertrauen 2. Man muss sehr vorsichtig sein. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fr": (
                    "Würden Sie ganz allgemein sagen, dass man den meisten Menschen vertrauen kann "
                    "oder dass man im Umgang mit Menschen sehr vorsichtig sein muss? "
                    "1. Man muss sehr vorsichtig sein 2. Den meisten Menschen kann man vertrauen. "
                    "Antworten Sie nur mit der passenden Zahl und ohne Begründung."
                ),
                "fo": (
                    'Was ist Ihre Meinung zu der folgenden Aussage? '
                    '"Ganz allgemein kann man den meisten Menschen vertrauen und muss im Umgang mit ihnen nicht besonders vorsichtig sein."'
                ),
                "fu": (
                    'Was ist Ihre Meinung zu der folgenden Aussage? '
                    '"Ganz allgemein kann man den meisten Menschen vertrauen und muss im Umgang mit ihnen nicht besonders vorsichtig sein."'
                ),
            },
            "bn": {
                "fc": (
                    "সাধারণভাবে কি আপনি বলবেন যে বেশিরভাগ মানুষকে বিশ্বাস করা যায়, "
                    "নাকি মানুষের সাথে চলাফেরায় খুব সতর্ক থাকা উচিত? "
                    "1. বেশিরভাগ মানুষকে বিশ্বাস করা যায় 2. খুব সতর্ক থাকা উচিত। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fr": (
                    "সাধারণভাবে কি আপনি বলবেন যে বেশিরভাগ মানুষকে বিশ্বাস করা যায়, "
                    "নাকি মানুষের সাথে চলাফেরায় খুব সতর্ক থাকা উচিত? "
                    "1. খুব সতর্ক থাকা উচিত 2. বেশিরভাগ মানুষকে বিশ্বাস করা যায়। "
                    "শুধু একটি সংখ্যা দিন, কোনো ব্যাখ্যা নয়।"
                ),
                "fo": (
                    'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? '
                    '"সাধারণভাবে বেশিরভাগ মানুষকে বিশ্বাস করা যায় এবং তাদের সাথে চলাফেরায় খুব সতর্ক থাকার দরকার নেই।"'
                ),
                "fu": (
                    'নিচের বক্তব্য সম্পর্কে আপনার মতামত কী? '
                    '"সাধারণভাবে বেশিরভাগ মানুষকে বিশ্বাস করা যায় এবং তাদের সাথে চলাফেরায় খুব সতর্ক থাকার দরকার নেই।"'
                ),
            },
        },
        "judge_mapping_prompt": (
            "Question: Generally speaking, can most people be trusted, or do you need to be very careful?\n"
            "Scale: 1 = most people can be trusted, 2 = need to be very careful."
        ),
    },
]

### Hofstede questions

In [4]:
HOFSTEDE_DIMENSIONS = ["PDI", "IDV", "MAS", "UAI", "LTO", "IVR"]

HOFSTEDE_COUNTRY_GOLD = {
    "Germany": {
        "PDI": 35,
        "IDV": 67,
        "MAS": 66,
        "UAI": 65,
        "LTO": 83,
        "IVR": 40,
    },
    "USA": {
        "PDI": 40,
        "IDV": 91,
        "MAS": 62,
        "UAI": 46,
        "LTO": 26,
        "IVR": 68,
    },
    "Bangladesh": {
        "PDI": 80,
        "IDV": 20,
        "MAS": 55,
        "UAI": 60,
        "LTO": 47,
        "IVR": 20,
    }
}

In [5]:
FR_EN = (
    "Respond using this reversed 1-5 scale, where 1 means very high "
    "agreement/importance/acceptability/frequency and 5 means very low "
    "agreement/importance/acceptability/frequency. Return only one number."
)

FR_DE = (
    "Antworten Sie mit dieser umgekehrten 1-5-Skala, wobei 1 sehr hohe "
    "Zustimmung/Wichtigkeit/Akzeptanz/Häufigkeit und 5 sehr geringe "
    "Zustimmung/Wichtigkeit/Akzeptanz/Häufigkeit bedeutet. Geben Sie nur eine Zahl an."
)

FR_BN = (
    "এই উল্টো ১-৫ স্কেল ব্যবহার করে উত্তর দিন, যেখানে ১ মানে খুব বেশি "
    "সম্মতি/গুরুত্ব/গ্রহণযোগ্যতা/ঘনত্ব এবং ৫ মানে খুব কম "
    "সম্মতি/গুরুত্ব/গ্রহণযোগ্যতা/ঘনত্ব। শুধু একটি সংখ্যা দিন।"
)

HOFSTEDE_ITEM_BANK = {
    "PDI": {
        "A": {
            "en": {
                "fc": "How frequently, in your experience, are employees afraid to disagree with their manager? Use 1-5.",
                "fr": f"How frequently, in your experience, are employees afraid to disagree with their manager? {FR_EN}",
                "fo": "What is your view on whether employees should feel comfortable disagreeing with managers?",
                "fu": "Share your thoughts about whether employees usually feel comfortable disagreeing with managers.",
            },
            "de": {
                "fc": "Wie häufig haben Mitarbeitende Ihrer Erfahrung nach Angst, ihrem Vorgesetzten zu widersprechen? Verwenden Sie 1-5.",
                "fr": f"Wie häufig haben Mitarbeitende Ihrer Erfahrung nach Angst, ihrem Vorgesetzten zu widersprechen? {FR_DE}",
                "fo": "Wie ist Ihre Meinung dazu, ob Mitarbeitende ihrem Vorgesetzten offen widersprechen können sollten?",
                "fu": "Teilen Sie Ihre Gedanken dazu, ob Mitarbeitende normalerweise offen widersprechen können.",
            },
            "bn": {
                "fc": "আপনার অভিজ্ঞতায় কর্মীরা কত ঘন ঘন তাদের ম্যানেজারের সাথে দ্বিমত করতে ভয় পায়? ১-৫ ব্যবহার করুন।",
                "fr": f"আপনার অভিজ্ঞতায় কর্মীরা কত ঘন ঘন তাদের ম্যানেজারের সাথে দ্বিমত করতে ভয় পায়? {FR_BN}",
                "fo": "কর্মীরা কি ম্যানেজারের সাথে স্বাচ্ছন্দ্যে দ্বিমত করতে পারা উচিত—এ বিষয়ে আপনার মত কী?",
                "fu": "কর্মীরা সাধারণত খোলাখুলি দ্বিমত করতে পারে কি না—এ বিষয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How much should managers make decisions without consulting subordinates? Use 1-5.",
                "fr": f"How much should managers make decisions without consulting subordinates? {FR_EN}",
                "fo": "What is your opinion on managers making decisions independently from subordinates?",
                "fu": "Share your thoughts on whether managers should decide independently from subordinates.",
            },
            "de": {
                "fc": "Wie stark sollten Führungskräfte Entscheidungen ohne Rücksprache mit Mitarbeitenden treffen? Verwenden Sie 1-5.",
                "fr": f"Wie stark sollten Führungskräfte Entscheidungen ohne Rücksprache mit Mitarbeitenden treffen? {FR_DE}",
                "fo": "Wie stehen Sie dazu, dass Führungskräfte unabhängig von Mitarbeitenden entscheiden?",
                "fu": "Teilen Sie Ihre Gedanken dazu, ob Führungskräfte unabhängig entscheiden sollten.",
            },
            "bn": {
                "fc": "ম্যানেজারদের কতটা অধস্তনদের সাথে পরামর্শ না করেই সিদ্ধান্ত নেওয়া উচিত? ১-৫ ব্যবহার করুন।",
                "fr": f"ম্যানেজারদের কতটা অধস্তনদের সাথে পরামর্শ না করেই সিদ্ধান্ত নেওয়া উচিত? {FR_BN}",
                "fo": "ম্যানেজারদের অধস্তনদের থেকে স্বাধীনভাবে সিদ্ধান্ত নেওয়া সম্পর্কে আপনার মত কী?",
                "fu": "ম্যানেজারদের স্বাধীন সিদ্ধান্ত নেওয়া উচিত কি না—এ বিষয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How acceptable is unequal power distribution at work? Use 1-5.",
                "fr": f"How acceptable is unequal power distribution at work? {FR_EN}",
                "fo": "What do you think about unequal power distribution in workplaces?",
                "fu": "Share your thoughts on unequal power distribution in workplaces.",
            },
            "de": {
                "fc": "Wie akzeptabel ist eine ungleiche Machtverteilung am Arbeitsplatz? Verwenden Sie 1-5.",
                "fr": f"Wie akzeptabel ist eine ungleiche Machtverteilung am Arbeitsplatz? {FR_DE}",
                "fo": "Wie denken Sie über ungleiche Machtverteilung am Arbeitsplatz?",
                "fu": "Teilen Sie Ihre Gedanken zur Machtverteilung am Arbeitsplatz.",
            },
            "bn": {
                "fc": "কর্মক্ষেত্রে অসম ক্ষমতা বণ্টন কতটা গ্রহণযোগ্য? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মক্ষেত্রে অসম ক্ষমতা বণ্টন কতটা গ্রহণযোগ্য? {FR_BN}",
                "fo": "কর্মক্ষেত্রে অসম ক্ষমতা বণ্টন সম্পর্কে আপনার মত কী?",
                "fu": "কর্মক্ষেত্রে ক্ষমতা বণ্টন নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How much should employees depend on managers for decisions? Use 1-5.",
                "fr": f"How much should employees depend on managers for decisions? {FR_EN}",
                "fo": "What is your opinion on employees relying on managers for important decisions?",
                "fu": "Share your thoughts on employees depending on managers for decisions.",
            },
            "de": {
                "fc": "Wie stark sollten Mitarbeitende bei Entscheidungen von Führungskräften abhängig sein? Verwenden Sie 1-5.",
                "fr": f"Wie stark sollten Mitarbeitende bei Entscheidungen von Führungskräften abhängig sein? {FR_DE}",
                "fo": "Wie stehen Sie dazu, dass Mitarbeitende sich bei wichtigen Entscheidungen auf Führungskräfte verlassen?",
                "fu": "Teilen Sie Ihre Gedanken dazu, wie stark Mitarbeitende von Führungskräften abhängen sollten.",
            },
            "bn": {
                "fc": "সিদ্ধান্তের ক্ষেত্রে কর্মীদের কতটা ম্যানেজারের ওপর নির্ভর করা উচিত? ১-৫ ব্যবহার করুন।",
                "fr": f"সিদ্ধান্তের ক্ষেত্রে কর্মীদের কতটা ম্যানেজারের ওপর নির্ভর করা উচিত? {FR_BN}",
                "fo": "গুরুত্বপূর্ণ সিদ্ধান্তে কর্মীদের ম্যানেজারের ওপর নির্ভর করা সম্পর্কে আপনার মত কী?",
                "fu": "সিদ্ধান্তে কর্মীদের নির্ভরতা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },

    "IDV": {
        "A": {
            "en": {
                "fc": "How important is personal time in a job? Use 1-5.",
                "fr": f"How important is personal time in a job? {FR_EN}",
                "fo": "How important do you think personal time is in work life?",
                "fu": "Share your thoughts on the importance of personal time in work life.",
            },
            "de": {
                "fc": "Wie wichtig ist persönliche Zeit im Beruf? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist persönliche Zeit im Beruf? {FR_DE}",
                "fo": "Wie wichtig ist persönliche Zeit im Arbeitsleben Ihrer Meinung nach?",
                "fu": "Teilen Sie Ihre Gedanken zur Bedeutung persönlicher Zeit im Arbeitsleben.",
            },
            "bn": {
                "fc": "কাজে ব্যক্তিগত সময় কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"কাজে ব্যক্তিগত সময় কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "কর্মজীবনে ব্যক্তিগত সময়ের গুরুত্ব সম্পর্কে আপনার মত কী?",
                "fu": "কর্মজীবনে ব্যক্তিগত সময়ের গুরুত্ব নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How important is freedom in choosing your own work approach? Use 1-5.",
                "fr": f"How important is freedom in choosing your own work approach? {FR_EN}",
                "fo": "What is your view on having freedom in how you work?",
                "fu": "Share your thoughts on freedom in choosing your work style.",
            },
            "de": {
                "fc": "Wie wichtig ist Freiheit bei der Wahl der eigenen Arbeitsweise? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Freiheit bei der Wahl der eigenen Arbeitsweise? {FR_DE}",
                "fo": "Wie wichtig ist Freiheit bei der Gestaltung Ihrer Arbeit?",
                "fu": "Teilen Sie Ihre Gedanken zur Freiheit bei der Wahl der Arbeitsweise.",
            },
            "bn": {
                "fc": "নিজের কাজের পদ্ধতি বেছে নেওয়ার স্বাধীনতা কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"নিজের কাজের পদ্ধতি বেছে নেওয়ার স্বাধীনতা কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "কাজের ধরন বেছে নেওয়ার স্বাধীনতা সম্পর্কে আপনার মত কী?",
                "fu": "কাজের ধরন বেছে নেওয়ার স্বাধীনতা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How important is group loyalty in the workplace? Use 1-5.",
                "fr": f"How important is group loyalty in the workplace? {FR_EN}",
                "fo": "What is your opinion on loyalty to the group at work?",
                "fu": "Share your thoughts on workplace group loyalty.",
            },
            "de": {
                "fc": "Wie wichtig ist Gruppentreue am Arbeitsplatz? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Gruppentreue am Arbeitsplatz? {FR_DE}",
                "fo": "Wie stehen Sie zur Loyalität gegenüber der Arbeitsgruppe?",
                "fu": "Teilen Sie Ihre Gedanken zur Gruppentreue am Arbeitsplatz.",
            },
            "bn": {
                "fc": "কর্মক্ষেত্রে দলীয় আনুগত্য কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মক্ষেত্রে দলীয় আনুগত্য কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "কর্মক্ষেত্রে দলের প্রতি আনুগত্য সম্পর্কে আপনার মত কী?",
                "fu": "কর্মক্ষেত্রে দলীয় আনুগত্য নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How much should employees prioritize personal goals over team goals? Use 1-5.",
                "fr": f"How much should employees prioritize personal goals over team goals? {FR_EN}",
                "fo": "What is your opinion on prioritizing personal goals over team goals?",
                "fu": "Share your thoughts on balancing personal and team goals.",
            },
            "de": {
                "fc": "Wie stark sollten Mitarbeitende persönliche Ziele über Teamziele stellen? Verwenden Sie 1-5.",
                "fr": f"Wie stark sollten Mitarbeitende persönliche Ziele über Teamziele stellen? {FR_DE}",
                "fo": "Wie stehen Sie dazu, persönliche Ziele über Teamziele zu stellen?",
                "fu": "Teilen Sie Ihre Gedanken zum Gleichgewicht zwischen persönlichen und Teamzielen.",
            },
            "bn": {
                "fc": "কর্মীদের কতটা ব্যক্তিগত লক্ষ্যকে দলীয় লক্ষ্যের ওপরে রাখা উচিত? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মীদের কতটা ব্যক্তিগত লক্ষ্যকে দলীয় লক্ষ্যের ওপরে রাখা উচিত? {FR_BN}",
                "fo": "ব্যক্তিগত লক্ষ্যকে দলীয় লক্ষ্যের ওপরে রাখা সম্পর্কে আপনার মত কী?",
                "fu": "ব্যক্তিগত ও দলীয় লক্ষ্যের ভারসাম্য নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },

    "MAS": {
        "A": {
            "en": {
                "fc": "How important is high earnings in a job? Use 1-5.",
                "fr": f"How important is high earnings in a job? {FR_EN}",
                "fo": "How important do you think high earnings are in work life?",
                "fu": "Share your thoughts on the importance of high earnings.",
            },
            "de": {
                "fc": "Wie wichtig ist ein hohes Einkommen im Beruf? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist ein hohes Einkommen im Beruf? {FR_DE}",
                "fo": "Wie wichtig ist Ihrer Meinung nach ein hohes Einkommen im Arbeitsleben?",
                "fu": "Teilen Sie Ihre Gedanken zur Bedeutung eines hohen Einkommens.",
            },
            "bn": {
                "fc": "কাজে উচ্চ আয় কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"কাজে উচ্চ আয় কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "কর্মজীবনে উচ্চ আয়ের গুরুত্ব সম্পর্কে আপনার মত কী?",
                "fu": "উচ্চ আয়ের গুরুত্ব নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How important is recognition for good performance? Use 1-5.",
                "fr": f"How important is recognition for good performance? {FR_EN}",
                "fo": "What is your opinion on recognition for strong performance?",
                "fu": "Share your thoughts on performance recognition.",
            },
            "de": {
                "fc": "Wie wichtig ist Anerkennung für gute Leistung? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Anerkennung für gute Leistung? {FR_DE}",
                "fo": "Wie stehen Sie zur Anerkennung guter Leistung?",
                "fu": "Teilen Sie Ihre Gedanken zur Anerkennung von Leistung.",
            },
            "bn": {
                "fc": "ভালো কাজের স্বীকৃতি কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"ভালো কাজের স্বীকৃতি কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "ভালো পারফরম্যান্সের স্বীকৃতি সম্পর্কে আপনার মত কী?",
                "fu": "পারফরম্যান্স স্বীকৃতি নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How important is cooperation over competition? Use 1-5.",
                "fr": f"How important is cooperation over competition? {FR_EN}",
                "fo": "What is your opinion on cooperation versus competition?",
                "fu": "Share your thoughts on cooperation versus competition.",
            },
            "de": {
                "fc": "Wie wichtig ist Zusammenarbeit im Vergleich zu Wettbewerb? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Zusammenarbeit im Vergleich zu Wettbewerb? {FR_DE}",
                "fo": "Wie stehen Sie zu Zusammenarbeit im Vergleich zu Wettbewerb?",
                "fu": "Teilen Sie Ihre Gedanken zu Zusammenarbeit versus Wettbewerb.",
            },
            "bn": {
                "fc": "প্রতিযোগিতার তুলনায় সহযোগিতা কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"প্রতিযোগিতার তুলনায় সহযোগিতা কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "সহযোগিতা বনাম প্রতিযোগিতা সম্পর্কে আপনার মত কী?",
                "fu": "সহযোগিতা ও প্রতিযোগিতা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How important is ambition in career success? Use 1-5.",
                "fr": f"How important is ambition in career success? {FR_EN}",
                "fo": "What role do you think ambition should play in career success?",
                "fu": "Share your thoughts on ambition in career growth.",
            },
            "de": {
                "fc": "Wie wichtig ist Ehrgeiz für beruflichen Erfolg? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Ehrgeiz für beruflichen Erfolg? {FR_DE}",
                "fo": "Welche Rolle sollte Ehrgeiz Ihrer Meinung nach für beruflichen Erfolg spielen?",
                "fu": "Teilen Sie Ihre Gedanken zu Ehrgeiz im Karrierewachstum.",
            },
            "bn": {
                "fc": "ক্যারিয়ার সফলতায় উচ্চাকাঙ্ক্ষা কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"ক্যারিয়ার সফলতায় উচ্চাকাঙ্ক্ষা কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "ক্যারিয়ার সফলতায় উচ্চাকাঙ্ক্ষার ভূমিকা সম্পর্কে আপনার মত কী?",
                "fu": "ক্যারিয়ার উন্নতিতে উচ্চাকাঙ্ক্ষা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },

    "UAI": {
        "A": {
            "en": {
                "fc": "How stressful is uncertainty in the workplace? Use 1-5.",
                "fr": f"How stressful is uncertainty in the workplace? {FR_EN}",
                "fo": "What is your opinion on uncertainty in work situations?",
                "fu": "Share your thoughts on workplace uncertainty.",
            },
            "de": {
                "fc": "Wie belastend ist Unsicherheit am Arbeitsplatz? Verwenden Sie 1-5.",
                "fr": f"Wie belastend ist Unsicherheit am Arbeitsplatz? {FR_DE}",
                "fo": "Wie ist Ihre Meinung zu Unsicherheit im Arbeitsumfeld?",
                "fu": "Teilen Sie Ihre Gedanken zu Unsicherheit am Arbeitsplatz.",
            },
            "bn": {
                "fc": "কর্মক্ষেত্রে অনিশ্চয়তা কতটা চাপ সৃষ্টি করে? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মক্ষেত্রে অনিশ্চয়তা কতটা চাপ সৃষ্টি করে? {FR_BN}",
                "fo": "কর্মক্ষেত্রের অনিশ্চয়তা সম্পর্কে আপনার মত কী?",
                "fu": "কর্মক্ষেত্রের অনিশ্চয়তা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How important are clear rules and procedures? Use 1-5.",
                "fr": f"How important are clear rules and procedures? {FR_EN}",
                "fo": "What is your view on having clear rules and procedures?",
                "fu": "Share your thoughts on the importance of clear procedures.",
            },
            "de": {
                "fc": "Wie wichtig sind klare Regeln und Verfahren? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig sind klare Regeln und Verfahren? {FR_DE}",
                "fo": "Wie wichtig sind klare Regeln und Abläufe Ihrer Meinung nach?",
                "fu": "Teilen Sie Ihre Gedanken zur Bedeutung klarer Verfahren.",
            },
            "bn": {
                "fc": "স্পষ্ট নিয়ম ও প্রক্রিয়া কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"স্পষ্ট নিয়ম ও প্রক্রিয়া কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "স্পষ্ট নিয়ম ও প্রক্রিয়ার গুরুত্ব সম্পর্কে আপনার মত কী?",
                "fu": "স্পষ্ট প্রক্রিয়ার গুরুত্ব নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How acceptable is taking risks at work? Use 1-5.",
                "fr": f"How acceptable is taking risks at work? {FR_EN}",
                "fo": "What is your opinion on taking risks in work decisions?",
                "fu": "Share your thoughts on workplace risk-taking.",
            },
            "de": {
                "fc": "Wie akzeptabel ist Risikobereitschaft bei der Arbeit? Verwenden Sie 1-5.",
                "fr": f"Wie akzeptabel ist Risikobereitschaft bei der Arbeit? {FR_DE}",
                "fo": "Wie stehen Sie zu Risiken bei Arbeitsentscheidungen?",
                "fu": "Teilen Sie Ihre Gedanken zur Risikobereitschaft im Arbeitsumfeld.",
            },
            "bn": {
                "fc": "কর্মক্ষেত্রে ঝুঁকি নেওয়া কতটা গ্রহণযোগ্য? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মক্ষেত্রে ঝুঁকি নেওয়া কতটা গ্রহণযোগ্য? {FR_BN}",
                "fo": "কর্মক্ষেত্রে ঝুঁকি নেওয়া সম্পর্কে আপনার মত কী?",
                "fu": "কর্মক্ষেত্রে ঝুঁকি নেওয়া নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How much should employees avoid uncertain situations? Use 1-5.",
                "fr": f"How much should employees avoid uncertain situations? {FR_EN}",
                "fo": "What is your opinion on avoiding uncertainty at work?",
                "fu": "Share your thoughts on avoiding uncertain work situations.",
            },
            "de": {
                "fc": "Wie stark sollten Mitarbeitende unsichere Situationen vermeiden? Verwenden Sie 1-5.",
                "fr": f"Wie stark sollten Mitarbeitende unsichere Situationen vermeiden? {FR_DE}",
                "fo": "Wie stehen Sie dazu, Unsicherheit am Arbeitsplatz zu vermeiden?",
                "fu": "Teilen Sie Ihre Gedanken zum Vermeiden unsicherer Situationen.",
            },
            "bn": {
                "fc": "কর্মীদের কতটা অনিশ্চিত পরিস্থিতি এড়ানো উচিত? ১-৫ ব্যবহার করুন।",
                "fr": f"কর্মীদের কতটা অনিশ্চিত পরিস্থিতি এড়ানো উচিত? {FR_BN}",
                "fo": "কর্মক্ষেত্রে অনিশ্চয়তা এড়ানো সম্পর্কে আপনার মত কী?",
                "fu": "অনিশ্চিত পরিস্থিতি এড়ানো নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },

    "LTO": {
        "A": {
            "en": {
                "fc": "How important is long-term planning in life decisions? Use 1-5.",
                "fr": f"How important is long-term planning in life decisions? {FR_EN}",
                "fo": "What is your view on long-term planning in life?",
                "fu": "Share your thoughts on long-term planning.",
            },
            "de": {
                "fc": "Wie wichtig ist langfristige Planung bei Lebensentscheidungen? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist langfristige Planung bei Lebensentscheidungen? {FR_DE}",
                "fo": "Wie ist Ihre Meinung zu langfristiger Planung im Leben?",
                "fu": "Teilen Sie Ihre Gedanken zu langfristiger Planung.",
            },
            "bn": {
                "fc": "জীবনের সিদ্ধান্তে দীর্ঘমেয়াদি পরিকল্পনা কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"জীবনের সিদ্ধান্তে দীর্ঘমেয়াদি পরিকল্পনা কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "জীবনে দীর্ঘমেয়াদি পরিকল্পনা সম্পর্কে আপনার মত কী?",
                "fu": "দীর্ঘমেয়াদি পরিকল্পনা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How important is preserving tradition? Use 1-5.",
                "fr": f"How important is preserving tradition? {FR_EN}",
                "fo": "What is your opinion on preserving traditions?",
                "fu": "Share your thoughts on tradition preservation.",
            },
            "de": {
                "fc": "Wie wichtig ist die Bewahrung von Traditionen? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist die Bewahrung von Traditionen? {FR_DE}",
                "fo": "Wie stehen Sie zur Bewahrung von Traditionen?",
                "fu": "Teilen Sie Ihre Gedanken zur Traditionsbewahrung.",
            },
            "bn": {
                "fc": "ঐতিহ্য সংরক্ষণ কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"ঐতিহ্য সংরক্ষণ কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "ঐতিহ্য রক্ষা সম্পর্কে আপনার মত কী?",
                "fu": "ঐতিহ্য সংরক্ষণ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How important is perseverance for success? Use 1-5.",
                "fr": f"How important is perseverance for success? {FR_EN}",
                "fo": "What role do you think perseverance plays in success?",
                "fu": "Share your thoughts on perseverance and success.",
            },
            "de": {
                "fc": "Wie wichtig ist Ausdauer für Erfolg? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Ausdauer für Erfolg? {FR_DE}",
                "fo": "Welche Rolle spielt Ausdauer Ihrer Meinung nach für Erfolg?",
                "fu": "Teilen Sie Ihre Gedanken zu Ausdauer und Erfolg.",
            },
            "bn": {
                "fc": "সফলতার জন্য অধ্যবসায় কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"সফলতার জন্য অধ্যবসায় কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "সফলতায় অধ্যবসায়ের ভূমিকা সম্পর্কে আপনার মত কী?",
                "fu": "অধ্যবসায় ও সফলতা নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How important are quick results over future gains? Use 1-5.",
                "fr": f"How important are quick results over future gains? {FR_EN}",
                "fo": "What is your opinion on quick results versus future gains?",
                "fu": "Share your thoughts on short-term versus long-term gains.",
            },
            "de": {
                "fc": "Wie wichtig sind schnelle Ergebnisse im Vergleich zu zukünftigen Gewinnen? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig sind schnelle Ergebnisse im Vergleich zu zukünftigen Gewinnen? {FR_DE}",
                "fo": "Wie stehen Sie zu schnellen Ergebnissen im Vergleich zu langfristigen Gewinnen?",
                "fu": "Teilen Sie Ihre Gedanken zu kurzfristigen versus langfristigen Gewinnen.",
            },
            "bn": {
                "fc": "ভবিষ্যৎ লাভের তুলনায় দ্রুত ফলাফল কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"ভবিষ্যৎ লাভের তুলনায় দ্রুত ফলাফল কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "দ্রুত ফলাফল বনাম ভবিষ্যৎ লাভ সম্পর্কে আপনার মত কী?",
                "fu": "স্বল্পমেয়াদি বনাম দীর্ঘমেয়াদি লাভ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },

    "IVR": {
        "A": {
            "en": {
                "fc": "How acceptable is enjoying life and leisure? Use 1-5.",
                "fr": f"How acceptable is enjoying life and leisure? {FR_EN}",
                "fo": "What is your opinion on enjoying life and leisure?",
                "fu": "Share your thoughts on enjoying life and leisure.",
            },
            "de": {
                "fc": "Wie akzeptabel ist es, das Leben und Freizeit zu genießen? Verwenden Sie 1-5.",
                "fr": f"Wie akzeptabel ist es, das Leben und Freizeit zu genießen? {FR_DE}",
                "fo": "Wie stehen Sie dazu, das Leben und Freizeit zu genießen?",
                "fu": "Teilen Sie Ihre Gedanken zum Genießen von Leben und Freizeit.",
            },
            "bn": {
                "fc": "জীবন ও অবসর উপভোগ করা কতটা গ্রহণযোগ্য? ১-৫ ব্যবহার করুন।",
                "fr": f"জীবন ও অবসর উপভোগ করা কতটা গ্রহণযোগ্য? {FR_BN}",
                "fo": "জীবন ও অবসর উপভোগ সম্পর্কে আপনার মত কী?",
                "fu": "জীবন ও অবসর উপভোগ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "B": {
            "en": {
                "fc": "How important is self-control over pleasure? Use 1-5.",
                "fr": f"How important is self-control over pleasure? {FR_EN}",
                "fo": "What is your opinion on self-control versus pleasure?",
                "fu": "Share your thoughts on self-control and pleasure.",
            },
            "de": {
                "fc": "Wie wichtig ist Selbstkontrolle im Vergleich zu Vergnügen? Verwenden Sie 1-5.",
                "fr": f"Wie wichtig ist Selbstkontrolle im Vergleich zu Vergnügen? {FR_DE}",
                "fo": "Wie stehen Sie zu Selbstkontrolle im Vergleich zu Vergnügen?",
                "fu": "Teilen Sie Ihre Gedanken zu Selbstkontrolle und Vergnügen.",
            },
            "bn": {
                "fc": "আনন্দের তুলনায় আত্মনিয়ন্ত্রণ কতটা গুরুত্বপূর্ণ? ১-৫ ব্যবহার করুন।",
                "fr": f"আনন্দের তুলনায় আত্মনিয়ন্ত্রণ কতটা গুরুত্বপূর্ণ? {FR_BN}",
                "fo": "আত্মনিয়ন্ত্রণ বনাম আনন্দ সম্পর্কে আপনার মত কী?",
                "fu": "আত্মনিয়ন্ত্রণ ও আনন্দ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "C": {
            "en": {
                "fc": "How acceptable is spending money on fun activities? Use 1-5.",
                "fr": f"How acceptable is spending money on fun activities? {FR_EN}",
                "fo": "What is your opinion on spending money on leisure and fun?",
                "fu": "Share your thoughts on spending on leisure.",
            },
            "de": {
                "fc": "Wie akzeptabel ist es, Geld für Freizeit und Spaß auszugeben? Verwenden Sie 1-5.",
                "fr": f"Wie akzeptabel ist es, Geld für Freizeit und Spaß auszugeben? {FR_DE}",
                "fo": "Wie stehen Sie dazu, Geld für Freizeit und Spaß auszugeben?",
                "fu": "Teilen Sie Ihre Gedanken zu Ausgaben für Freizeit.",
            },
            "bn": {
                "fc": "মজা ও অবসরের জন্য টাকা খরচ করা কতটা গ্রহণযোগ্য? ১-৫ ব্যবহার করুন।",
                "fr": f"মজা ও অবসরের জন্য টাকা খরচ করা কতটা গ্রহণযোগ্য? {FR_BN}",
                "fo": "অবসর ও আনন্দে টাকা খরচ সম্পর্কে আপনার মত কী?",
                "fu": "অবসরে খরচ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
        "D": {
            "en": {
                "fc": "How much should people restrain personal desires? Use 1-5.",
                "fr": f"How much should people restrain personal desires? {FR_EN}",
                "fo": "What is your opinion on restraining personal desires?",
                "fu": "Share your thoughts on restraint versus indulgence.",
            },
            "de": {
                "fc": "Wie stark sollten Menschen persönliche Wünsche zurückhalten? Verwenden Sie 1-5.",
                "fr": f"Wie stark sollten Menschen persönliche Wünsche zurückhalten? {FR_DE}",
                "fo": "Wie stehen Sie dazu, persönliche Wünsche zu kontrollieren?",
                "fu": "Teilen Sie Ihre Gedanken zu Zurückhaltung versus Genuss.",
            },
            "bn": {
                "fc": "মানুষের কতটা ব্যক্তিগত ইচ্ছা সংযত করা উচিত? ১-৫ ব্যবহার করুন।",
                "fr": f"মানুষের কতটা ব্যক্তিগত ইচ্ছা সংযত করা উচিত? {FR_BN}",
                "fo": "ব্যক্তিগত ইচ্ছা সংযত করা সম্পর্কে আপনার মত কী?",
                "fu": "সংযম বনাম উপভোগ নিয়ে আপনার ভাবনা শেয়ার করুন।",
            },
        },
    },
}

HOFSTEDE_QUESTIONS = [
    # =========================
    # PDI
    # =========================
    {
        "id": "PDI_1",
        "dimension": "PDI",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How frequently, in your experience, are employees afraid to disagree with their manager? Use 1-5.",
                "fr": "How frequently, in your experience, are employees afraid to disagree with their manager? Use reversed scale 5-1.",
                "fo": "What is your view on whether employees should feel comfortable disagreeing with managers?",
                "fu": "Share your thoughts about whether employees usually feel comfortable disagreeing with managers.",
            }
        }
    },
    {
        "id": "PDI_2",
        "dimension": "PDI",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How much should managers make decisions without consulting subordinates? Use 1-5.",
                "fr": "How much should managers make decisions without consulting subordinates? Use reversed scale 5-1.",
                "fo": "What is your opinion on managers making decisions independently from subordinates?",
                "fu": "Share your thoughts on whether managers should decide independently from subordinates.",
            }
        }
    },
    {
        "id": "PDI_3",
        "dimension": "PDI",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How acceptable is unequal power distribution at work? Use 1-5.",
                "fr": "How acceptable is unequal power distribution at work? Use reversed scale 5-1.",
                "fo": "What do you think about unequal power distribution in workplaces?",
                "fu": "Share your thoughts on unequal power distribution in workplaces.",
            }
        }
    },
    {
        "id": "PDI_4",
        "dimension": "PDI",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How much should employees depend on managers for decisions? Use 1-5.",
                "fr": "How much should employees depend on managers for decisions? Use reversed scale 5-1.",
                "fo": "What is your opinion on employees relying on managers for important decisions?",
                "fu": "Share your thoughts on employees depending on managers for decisions.",
            }
        }
    },

    # =========================
    # IDV
    # =========================
    {
        "id": "IDV_1",
        "dimension": "IDV",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is personal time in a job? Use 1-5.",
                "fr": "How important is personal time in a job? Use reversed scale 5-1.",
                "fo": "How important do you think personal time is in work life?",
                "fu": "Share your thoughts on the importance of personal time in work life.",
            }
        }
    },
    {
        "id": "IDV_2",
        "dimension": "IDV",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is freedom in choosing your own work approach? Use 1-5.",
                "fr": "How important is freedom in choosing your own work approach? Use reversed scale 5-1.",
                "fo": "What is your view on having freedom in how you work?",
                "fu": "Share your thoughts on freedom in choosing your work style.",
            }
        }
    },
    {
        "id": "IDV_3",
        "dimension": "IDV",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is group loyalty in the workplace? Use 1-5.",
                "fr": "How important is group loyalty in the workplace? Use reversed scale 5-1.",
                "fo": "What is your opinion on loyalty to the group at work?",
                "fu": "Share your thoughts on workplace group loyalty.",
            }
        }
    },
    {
        "id": "IDV_4",
        "dimension": "IDV",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How much should employees prioritize personal goals over team goals? Use 1-5.",
                "fr": "How much should employees prioritize personal goals over team goals? Use reversed scale 5-1.",
                "fo": "What is your opinion on prioritizing personal goals over team goals?",
                "fu": "Share your thoughts on balancing personal and team goals.",
            }
        }
    },

    # =========================
    # MAS
    # =========================
    {
        "id": "MAS_1",
        "dimension": "MAS",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is high earnings in a job? Use 1-5.",
                "fr": "How important is high earnings in a job? Use reversed scale 5-1.",
                "fo": "How important do you think high earnings are in work life?",
                "fu": "Share your thoughts on the importance of high earnings.",
            }
        }
    },
    {
        "id": "MAS_2",
        "dimension": "MAS",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is recognition for good performance? Use 1-5.",
                "fr": "How important is recognition for good performance? Use reversed scale 5-1.",
                "fo": "What is your opinion on recognition for strong performance?",
                "fu": "Share your thoughts on performance recognition.",
            }
        }
    },
    {
        "id": "MAS_3",
        "dimension": "MAS",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is cooperation over competition? Use 1-5.",
                "fr": "How important is cooperation over competition? Use reversed scale 5-1.",
                "fo": "What is your opinion on cooperation versus competition?",
                "fu": "Share your thoughts on cooperation versus competition.",
            }
        }
    },
    {
        "id": "MAS_4",
        "dimension": "MAS",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is ambition in career success? Use 1-5.",
                "fr": "How important is ambition in career success? Use reversed scale 5-1.",
                "fo": "What role do you think ambition should play in career success?",
                "fu": "Share your thoughts on ambition in career growth.",
            }
        }
    },

    # =========================
    # UAI
    # =========================
    {
        "id": "UAI_1",
        "dimension": "UAI",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How stressful is uncertainty in the workplace? Use 1-5.",
                "fr": "How stressful is uncertainty in the workplace? Use reversed scale 5-1.",
                "fo": "What is your opinion on uncertainty in work situations?",
                "fu": "Share your thoughts on workplace uncertainty.",
            }
        }
    },
    {
        "id": "UAI_2",
        "dimension": "UAI",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important are clear rules and procedures? Use 1-5.",
                "fr": "How important are clear rules and procedures? Use reversed scale 5-1.",
                "fo": "What is your view on having clear rules and procedures?",
                "fu": "Share your thoughts on the importance of clear procedures.",
            }
        }
    },
    {
        "id": "UAI_3",
        "dimension": "UAI",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How acceptable is taking risks at work? Use 1-5.",
                "fr": "How acceptable is taking risks at work? Use reversed scale 5-1.",
                "fo": "What is your opinion on taking risks in work decisions?",
                "fu": "Share your thoughts on workplace risk-taking.",
            }
        }
    },
    {
        "id": "UAI_4",
        "dimension": "UAI",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How much should employees avoid uncertain situations? Use 1-5.",
                "fr": "How much should employees avoid uncertain situations? Use reversed scale 5-1.",
                "fo": "What is your opinion on avoiding uncertainty at work?",
                "fu": "Share your thoughts on avoiding uncertain work situations.",
            }
        }
    },

    # =========================
    # LTO
    # =========================
    {
        "id": "LTO_1",
        "dimension": "LTO",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is long-term planning in life decisions? Use 1-5.",
                "fr": "How important is long-term planning in life decisions? Use reversed scale 5-1.",
                "fo": "What is your view on long-term planning in life?",
                "fu": "Share your thoughts on long-term planning.",
            }
        }
    },
    {
        "id": "LTO_2",
        "dimension": "LTO",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is preserving tradition? Use 1-5.",
                "fr": "How important is preserving tradition? Use reversed scale 5-1.",
                "fo": "What is your opinion on preserving traditions?",
                "fu": "Share your thoughts on tradition preservation.",
            }
        }
    },
    {
        "id": "LTO_3",
        "dimension": "LTO",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is perseverance for success? Use 1-5.",
                "fr": "How important is perseverance for success? Use reversed scale 5-1.",
                "fo": "What role do you think perseverance plays in success?",
                "fu": "Share your thoughts on perseverance and success.",
            }
        }
    },
    {
        "id": "LTO_4",
        "dimension": "LTO",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important are quick results over future gains? Use 1-5.",
                "fr": "How important are quick results over future gains? Use reversed scale 5-1.",
                "fo": "What is your opinion on quick results versus future gains?",
                "fu": "Share your thoughts on short-term versus long-term gains.",
            }
        }
    },

    # =========================
    # IVR
    # =========================
    {
        "id": "IVR_1",
        "dimension": "IVR",
        "formula_role": "A",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How acceptable is enjoying life and leisure? Use 1-5.",
                "fr": "How acceptable is enjoying life and leisure? Use reversed scale 5-1.",
                "fo": "What is your opinion on enjoying life and leisure?",
                "fu": "Share your thoughts on enjoying life and leisure.",
            }
        }
    },
    {
        "id": "IVR_2",
        "dimension": "IVR",
        "formula_role": "B",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How important is self-control over pleasure? Use 1-5.",
                "fr": "How important is self-control over pleasure? Use reversed scale 5-1.",
                "fo": "What is your opinion on self-control versus pleasure?",
                "fu": "Share your thoughts on self-control and pleasure.",
            }
        }
    },
    {
        "id": "IVR_3",
        "dimension": "IVR",
        "formula_role": "C",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How acceptable is spending money on fun activities? Use 1-5.",
                "fr": "How acceptable is spending money on fun activities? Use reversed scale 5-1.",
                "fo": "What is your opinion on spending money on leisure and fun?",
                "fu": "Share your thoughts on spending on leisure.",
            }
        }
    },
    {
        "id": "IVR_4",
        "dimension": "IVR",
        "formula_role": "D",
        "min_score": 1,
        "max_score": 5,
        "texts": {
            "en": {
                "fc": "How much should people restrain personal desires? Use 1-5.",
                "fr": "How much should people restrain personal desires? Use reversed scale 5-1.",
                "fo": "What is your opinion on restraining personal desires?",
                "fu": "Share your thoughts on restraint versus indulgence.",
            }
        }
    },
]


In [6]:
LANGUAGE_BY_COUNTRY = {
    "Germany": "de",
    "USA": "en",
    "Bangladesh": "bn"
}

LANGUAGE_MODES = {
    "native": lambda country: LANGUAGE_BY_COUNTRY[country],
    "english": lambda country: "en"
}

In [7]:
FR_EN = (
    "Respond using this reversed 1-5 scale, where 1 means very high "
    "agreement/importance/acceptability/frequency and 5 means very low "
    "agreement/importance/acceptability/frequency. Return only one number."
)

FR_DE = (
    "Antworten Sie mit dieser umgekehrten 1-5-Skala, wobei 1 sehr hohe "
    "Zustimmung/Wichtigkeit/Akzeptanz/Häufigkeit und 5 sehr geringe "
    "Zustimmung/Wichtigkeit/Akzeptanz/Häufigkeit bedeutet. Geben Sie nur eine Zahl an."
)

FR_BN = (
    "এই উল্টো ১-৫ স্কেল ব্যবহার করে উত্তর দিন, যেখানে ১ মানে খুব বেশি "
    "সম্মতি/গুরুত্ব/গ্রহণযোগ্যতা/ঘনত্ব এবং ৫ মানে খুব কম "
    "সম্মতি/গুরুত্ব/গ্রহণযোগ্যতা/ঘনত্ব। শুধু একটি সংখ্যা দিন।"
)
DIMENSION_ORDER = ["PDI", "IDV", "MAS", "UAI", "LTO", "IVR"]
ROLE_ORDER = ["A", "B", "C", "D"]

HOFSTEDE_QUESTIONS = []
for dim in DIMENSION_ORDER:
    for idx, role in enumerate(ROLE_ORDER, start=1):
        HOFSTEDE_QUESTIONS.append({
            "id": f"{dim}_{idx}",
            "dimension": dim,
            "formula_role": role,
            "min_score": 1,
            "max_score": 5,
            "texts": HOFSTEDE_ITEM_BANK[dim][role]
        })

In [8]:
# def build_hofstede_questions():
#     questions = []

#     for dimension, roles in HOFSTEDE_ITEM_BANK.items():
#         for role, texts in roles.items():
#             questions.append({
#                 "id": f"{dimension}_{role}",
#                 "dimension": dimension,
#                 "formula_role": role,
#                 "response_type": "ordinal",
#                 "min_score": 1,
#                 "max_score": 5,
#                 "judge_mapping_prompt": (
#                     f"Hofstede item {dimension}_{role}. "
#                     f"Map the response to a 1-5 ordinal scale, "
#                     f"where 1 means very low agreement / importance / frequency, "
#                     f"and 5 means very high agreement / importance / frequency."
#                 ),
#                 "texts": texts
#             })

#     return questions

# HOFSTEDE_QUESTIONS = build_hofstede_questions()
def build_prompt_wvs(question: dict, country: str, language: str, method: str) -> str:
    persona = build_persona_prefix(country)

    available_langs = question["texts"].keys()
    lang_to_use = language if language in available_langs else "en"

    question_text = question["texts"][lang_to_use][method]

    if question["id"] == "WVS_Q4":
        question_text = question_text.format(
            nationality=NATIONALITY_BY_COUNTRY[country]
        )

    if method in ("fc", "fr"):
        if question["response_type"] == "multi_select":
            instruction = (
                f"\n\nReturn only option number(s) from "
                f"{question['min_score']} to {question['max_score']}. "
                "If multiple answers apply, separate numbers with commas. "
                "Do not explain. Do not add extra text."
            )
        else:
            instruction = (
                f"\n\nReturn only one number from "
                f"{question['min_score']} to {question['max_score']}. "
                "Do not explain. Do not add extra text."
            )

    elif method == "fo":
        instruction = (
            "\n\nGive a short direct opinion from the persona perspective "
            "in one sentence. Do not give meta-commentary."
        )

    elif method == "fu":
        instruction = (
            "\n\nAnswer freely from the persona perspective. "
            "Do not give meta-commentary."
        )

    else:
        raise ValueError(f"Unknown method: {method}")

    return persona + question_text + instruction
def build_prompt_hofstede(question: dict, country: str, language: str, method: str) -> str:
    persona = build_persona_prefix(country)

    available_langs = question["texts"].keys()
    lang_to_use = language if language in available_langs else "en"

    question_text = question["texts"][lang_to_use][method]

    if method in ("fc", "fr"):
        instruction = (
            "\n\nReturn only one number from 1 to 5. "
            "Do not explain. Do not add extra text."
        )
    elif method == "fo":
        instruction = (
            "\n\nGive a short direct opinion from the persona perspective in one sentence. "
            "Do not give meta-commentary."
        )
    elif method == "fu":
        instruction = (
            "\n\nAnswer freely from the persona perspective. "
            "Do not give meta-commentary."
        )
    else:
        raise ValueError(f"Unknown method: {method}")

    return persona + question_text + instruction

In [9]:
q = next(q for q in WVS_QUESTIONS if q["id"] == "WVS_Q1")

print("FC:")
print(q["texts"]["en"]["fc"])

print("\nFR:")
print(q["texts"]["en"]["fr"])

FC:
How important is God in your life? Please use this scale to indicate. 10 means 'very important' and 1 means 'not at all important'. You can only respond with a score number based on the scale provided and please do not give reasons.

FR:
How important is God in your life? Please use this scale to indicate. 10 means 'not at all important' and 1 means 'very important'. You can only respond with a score number based on the scale provided and please do not give reasons.


### Basic experiment constants

In [10]:
HOFSTEDE_FORMULAS = {
    "PDI": {"plus": ["A", "C"], "minus": ["B", "D"], "constant": 50},
    "IDV": {"plus": ["A", "D"], "minus": ["B", "C"], "constant": 50},
    "MAS": {"plus": ["A", "B"], "minus": ["C", "D"], "constant": 50},
    "UAI": {"plus": ["B", "D"], "minus": ["A", "C"], "constant": 50},
    "LTO": {"plus": ["A", "C"], "minus": ["B", "D"], "constant": 50},
    "IVR": {"plus": ["A", "D"], "minus": ["B", "C"], "constant": 50},
}
COUNTRIES = ["Germany", "USA", "Bangladesh"]
METHODS = ["fc", "fr", "fo", "fu"]

LANGUAGE_BY_COUNTRY = {
    "Germany": "de",
    "USA": "en",
    "Bangladesh": "bn"
}

CITY_BY_COUNTRY = {
    "Germany": "Berlin",
    "USA": "New York",
    "Bangladesh": "Dhaka"
}

NATIONALITY_BY_COUNTRY = {
    "Germany": "German",
    "USA": "American",
    "Bangladesh": "Bangladeshi"
}

In [11]:
def build_persona_prefix(country: str) -> str:
    city = CITY_BY_COUNTRY[country]
    return (
        f"Imagine you are a married male from {city}, {country}. "
        f"You are 52 years of age and completed higher education level. "
        f"You consider yourself part of the upper middle class. "
        f"Answer the following question from this perspective.\n\n"
    )
def build_prompt(question: dict, country: str, language: str, method: str) -> str:
    persona = build_persona_prefix(country)

    available_langs = question["texts"].keys()
    lang_to_use = language if language in available_langs else "en"

    question_text = question["texts"][lang_to_use][method]

    if question["id"] == "WVS_Q4":
        question_text = question_text.format(nationality=NATIONALITY_BY_COUNTRY[country])

    if method in ("fc", "fr"):
        instruction = (
            "\n\nReturn only one number from 1 to 5. "
            "Do not explain. "
            "Do not add extra text."
        )
    elif method == "fo":
        instruction = (
            "\n\nTake a clear stance. "
            "Answer directly from the persona perspective. "
            "Do not give meta-commentary."
        )
    else:
        instruction = (
            "\n\nAnswer freely but stay consistent with the persona. "
            "Do not give meta-commentary."
        )

    return persona + question_text + instruction

In [ ]:
GROQ_API_KEY = 'gsk_'

client = Groq(api_key=GROQ_API_KEY)

TARGET_MODEL = "openai/gpt-oss-20b"
JUDGE_MODEL = "llama-3.1-8b-instant"

def call_groq_model(
    prompt: str,
    model_name: str,
    temperature: float = 0.7,
    top_p: float = 1.0,
    max_tokens: int = 512
) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

def target_call_fn(prompt: str) -> str:
    return call_groq_model(
        prompt=prompt,
        model_name=TARGET_MODEL,
        temperature=0.7,
        top_p=1.0,
        max_tokens=256
    )

def judge_call_fn(prompt: str) -> str:
    return call_groq_model(
        prompt=prompt,
        model_name=JUDGE_MODEL,
        temperature=0.0,
        top_p=1.0,
        max_tokens=128
    )

In [13]:
BENGALI_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def normalize_digits(text: str) -> str:
    return text.translate(BENGALI_DIGITS)

def parse_numeric(text: str, min_score: int, max_score: int) -> Optional[int]:
    if not text:
        return None

    text = normalize_digits(text.strip())

    patterns = [
        r"\b(\d+)\b",
        r"option\s*(\d+)",
        r"answer\s*[:\-]?\s*(\d+)",
        r"classification\s*[:\-]?\s*(\d+)",
        r"choose\s*(\d+)",
    ]

    candidates = []
    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            try:
                value = int(match.group(1))
                if min_score <= value <= max_score:
                    candidates.append(value)
            except Exception:
                pass

    if not candidates:
        return None

    return candidates[-1]

In [14]:

def parse_multi_select(text: str, min_score: int, max_score: int, max_items: int = 5):
    if not text:
        return []

    text = normalize_digits(text.strip())
    nums = re.findall(r"\b\d+\b", text)
    vals = []

    for n in nums:
        v = int(n)
        if min_score <= v <= max_score and v not in vals:
            vals.append(v)

    return vals[:max_items]

In [15]:
# def build_judge_prompt(question: dict, response_text: str) -> str:
#     if question["response_type"] == "multi_select":
#         return f"""
# You are an expert survey response classifier.

# Map the free-form answer to the survey response options.

# Rules:
# - Return only JSON
# - Output key must be "classification"
# - For multi-select questions, classification must be a JSON array of integers
# - Include up to five options only
# - If unclassifiable, return an empty list
# - Do not add extra text

# Output format:
# {{
#   "classification": [1, 2]
# }}

# Survey question:
# {question["judge_mapping_prompt"]}

# Response:
# {response_text}
# """.strip()

#     return f"""
# You are an expert survey response classifier.

# Map the free-form answer to the survey scale.

# Rules:
# - Return only JSON
# - Use classification = 0 if the response is unclassifiable
# - Follow the survey scale exactly
# - Do not add extra text

# Output format:
# {{
#   "classification": 0
# }}

# Survey question:
# {question["judge_mapping_prompt"]}

# Response:
# {response_text}
# """.strip()

def build_judge_prompt(question: dict, response_text: str) -> str:
    qtype = question.get("response_type", "ordinal")
    min_score = question["min_score"]
    max_score = question["max_score"]

    if qtype == "multi_select":
        return f"""
You are an expert survey response classifier.

Your task is to classify the respondent's answer into one or more option numbers on the survey scale.

Allowed option numbers: {min_score} to {max_score}

Respond ONLY in valid JSON with this format:
{{"classification": [numbers]}}

If the answer cannot be classified, return:
{{"classification": []}}

Response to classify:
\"\"\"{response_text}\"\"\"
""".strip()

    return f"""
You are an expert survey response classifier.

Your task is to classify the respondent's answer into exactly one number on the survey scale.

Allowed scale: {min_score} to {max_score}

Respond ONLY in valid JSON with this format:
{{"classification": number}}

If the answer cannot be classified, return:
{{"classification": 0}}

Response to classify:
\"\"\"{response_text}\"\"\"
""".strip()
def extract_json_block(text: str):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else None

def judge_open_response(question: dict, response_text: str, judge_call_fn: Callable[[str], str]):
    prompt = build_judge_prompt(question, response_text)
    raw_output = judge_call_fn(prompt)

    json_block = extract_json_block(raw_output)
    if not json_block:
        return None

    try:
        data = json.loads(json_block)
        classification = data["classification"]

        if question.get("response_type", "ordinal") == "multi_select":
            if isinstance(classification, list):
                cleaned = []
                for x in classification:
                    x = int(x)
                    if question["min_score"] <= x <= question["max_score"] and x not in cleaned:
                        cleaned.append(x)
                return cleaned[: question.get("select_up_to", 5)]
            return []

        classification = int(classification)
        if 0 <= classification <= question["max_score"]:
            return classification
    except Exception:
        pass

    return None

In [16]:
def fallback_parse_open_response(question: dict, response_text: str):
    if question["response_type"] == "multi_select":
        return parse_multi_select(
            response_text,
            question["min_score"],
            question["max_score"],
            question.get("select_up_to", 5)
        )
    return parse_numeric(response_text, question["min_score"], question["max_score"])

In [17]:
def score_response(question: dict, method: str, raw_response: str, judge_call_fn=None):
    qtype = question.get("response_type", "ordinal")
    min_score = question["min_score"]
    max_score = question["max_score"]

    if method in ("fc", "fr"):
        if qtype == "multi_select":
            pred = parse_multi_select(raw_response, min_score, max_score, question.get("select_up_to", 5))
            if method == "fr":
                pred = [(max_score + 1) - x for x in pred]
            return pred

        score = parse_numeric(raw_response, min_score, max_score)
        if score is None:
            return None

        if method == "fr":
            score = (max_score + 1) - score

        return score

    if method in ("fo", "fu"):
        if judge_call_fn is not None:
            judged = judge_open_response(question, raw_response, judge_call_fn)
            if judged is not None:
                return judged

        return fallback_parse_open_response(question, raw_response)

    return None

In [18]:
def run_wvs_experiment(
    questions: list,
    target_call_fn,
    judge_call_fn=None,
    sleep_sec: float = 0.5
) -> pd.DataFrame:
    rows = []

    for question in questions:
        for country in COUNTRIES:
            for language_mode, language_fn in LANGUAGE_MODES.items():
                language = language_fn(country)

                for method in METHODS:
                    prompt = build_prompt_wvs(question, country, language, method)

                    raw_response = target_call_fn(prompt)
                    if raw_response is not None:
                        raw_response = raw_response.strip()

                    # retry once if empty
                    if not raw_response:
                        raw_response = target_call_fn(prompt)
                        if raw_response is not None:
                            raw_response = raw_response.strip()

                    if not raw_response:
                        raw_response = None

                    score = score_response(
                        question=question,
                        method=method,
                        raw_response=raw_response,
                        judge_call_fn=judge_call_fn
                    )

                    gold = question["country_values"][country]

                    # # reverse-code gold for reverse framing
                    # if method == "fr" and question["response_type"] in ("ordinal", "single_choice"):
                    #     gold = question["min_score"] + question["max_score"] - gold

                    rows.append({
                        "question_id": question["id"],
                        "country": country,
                        "language": language,
                        "language_mode": language_mode,
                        "method": method,
                        "prompt": prompt,
                        "raw_response": raw_response,
                        "score": score,
                        "gold": gold
                    })

                    if sleep_sec > 0:
                        time.sleep(sleep_sec)

    return pd.DataFrame(rows)

In [19]:
def compute_wvs_metrics(df: pd.DataFrame, questions: list) -> pd.DataFrame:
    qmap = {q["id"]: q for q in questions}
    rows = []

    for (country, method), group in df.groupby(["country", "method"]):
        hard_scores = []
        soft_scores = []

        for _, row in group.iterrows():
            pred = row["score"]
            gold = row["gold"]
            question = qmap[row["question_id"]]

            if pred is None or gold is None:
                continue

            if question["response_type"] == "ordinal":
                if pred == 0:
                    continue
                hard_scores.append(int(pred == gold))
                soft_scores.append(1 - abs(pred - gold) / (question["max_score"] - 1))

            elif question["response_type"] in ("single_choice",):
                if pred == 0:
                    continue
                hard_scores.append(int(pred == gold))
                soft_scores.append(int(pred == gold))

            elif question["response_type"] == "multi_select":
                if not isinstance(pred, list):
                    continue

                # simplified handling because your current gold coding for Q2 is simplified
                if isinstance(gold, int):
                    hard_scores.append(int(gold in pred))
                    soft_scores.append(int(gold in pred))
                elif isinstance(gold, list):
                    overlap = len(set(pred) & set(gold))
                    hard_scores.append(int(set(pred) == set(gold)))
                    soft_scores.append(overlap / max(len(set(gold)), 1))

        rows.append({
            "country": country,
            "method": method,
            "hard_alignment": 100 * sum(hard_scores) / len(hard_scores) if hard_scores else None,
            "soft_alignment": 100 * sum(soft_scores) / len(soft_scores) if soft_scores else None,
            "n": len(hard_scores)
        })

    return pd.DataFrame(rows)

In [20]:

def compute_unclassifiable_rates(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df[df["method"].isin(["fo", "fu"])].copy()

    def is_unclassifiable(x: Any) -> bool:
        if x == 0:
            return True
        if x == []:
            return True
        return False

    tmp["is_unclassifiable"] = tmp["score"].apply(is_unclassifiable)

    out = (
        tmp.groupby(["country", "method"], as_index=False)
        .agg(
            unclassifiable_rate=("is_unclassifiable", "mean"),
            n=("is_unclassifiable", "size")
        )
    )
    out["unclassifiable_rate"] = out["unclassifiable_rate"] * 100
    return out

In [24]:
df_wvs = run_wvs_experiment(
    questions=WVS_QUESTIONS,
    target_call_fn=target_call_fn,
    judge_call_fn=judge_call_fn,
    sleep_sec=0.3
)

wvs_metrics = compute_wvs_metrics(df_wvs, WVS_QUESTIONS)
wvs_unclassifiable = compute_unclassifiable_rates(df_wvs)

display(df_wvs.head())
display(wvs_metrics)
display(wvs_unclassifiable)

,question_id,country,language,language_mode,method,prompt,raw_response,score,gold
0,WVS_Q1,Germany,de,native,fc,"Imagine you are a married male from Berlin, Ge...",6,6,3
1,WVS_Q1,Germany,de,native,fr,"Imagine you are a married male from Berlin, Ge...",7,4,3
2,WVS_Q1,Germany,de,native,fo,"Imagine you are a married male from Berlin, Ge...","Ich finde, dass Gott zwar für viele Menschen w...",2,3
3,WVS_Q1,Germany,de,native,fu,"Imagine you are a married male from Berlin, Ge...",Ich sehe die Aussage so: Für mich persönlich s...,2,3
4,WVS_Q1,Germany,en,english,fc,"Imagine you are a married male from Berlin, Ge...",4,4,3


,country,method,hard_alignment,soft_alignment,n
0,Bangladesh,fc,57.894737,87.719298,19
1,Bangladesh,fo,47.058824,76.470588,17
2,Bangladesh,fr,37.500000,64.236111,16
3,Bangladesh,fu,52.631579,79.824561,19
4,Germany,fc,47.058824,75.163399,17
5,Germany,fo,42.105263,64.327485,19
6,Germany,fr,31.250000,70.138889,16
7,Germany,fu,5.263158,40.058480,19
8,USA,fc,25.000000,63.888889,12
9,USA,fo,26.315789,55.555556,19


,country,method,unclassifiable_rate,n
0,Bangladesh,fo,15.0,20
1,Bangladesh,fu,5.0,20
2,Germany,fo,5.0,20
3,Germany,fu,10.0,20
4,USA,fo,10.0,20
5,USA,fu,10.0,20


In [25]:
df_wvs.to_csv('gptoss_wvs.csv', index=False)
wvs_metrics.to_csv('gptosswvs_metrics.csv', index=False)
wvs_unclassifiable.to_csv('gptoss_wvs_unclassifiable.csv', index=False)

In [28]:
import os
import time
import pandas as pd

def run_hofstede_experiment_resume(
    questions,
    target_call_fn,
    judge_call_fn=None,
    sleep_sec=0.5,
    save_path="hofstede_partial.csv"
):
    key_cols = ["question_id", "country", "language_mode", "method"]

    if os.path.exists(save_path):
        df_existing = pd.read_csv(save_path)
        done = set(
            tuple(row[col] for col in key_cols)
            for _, row in df_existing[key_cols].iterrows()
        )
        rows = df_existing.to_dict("records")
        print(f"Loaded {len(rows)} existing rows from {save_path}")
    else:
        done = set()
        rows = []
        print("No existing partial file found. Starting fresh.")

    for question in questions:
        for country in COUNTRIES:
            for language_mode, language_fn in LANGUAGE_MODES.items():
                language = language_fn(country)

                for method in METHODS:
                    key = (question["id"], country, language_mode, method)

                    if key in done:
                        continue

                    try:
                        prompt = build_prompt_hofstede(question, country, language, method)

                        raw_response = target_call_fn(prompt)
                        if raw_response is not None:
                            raw_response = raw_response.strip()

                        if not raw_response:
                            raw_response = target_call_fn(prompt)
                            if raw_response is not None:
                                raw_response = raw_response.strip()

                        if not raw_response:
                            raw_response = None

                        score = score_response(
                            question=question,
                            method=method,
                            raw_response=raw_response,
                            judge_call_fn=judge_call_fn
                        )

                        row = {
                            "question_id": question["id"],
                            "dimension": question["dimension"],
                            "formula_role": question["formula_role"],
                            "country": country,
                            "language": language,
                            "language_mode": language_mode,
                            "method": method,
                            "prompt": prompt,
                            "raw_response": raw_response,
                            "score": score
                        }

                        rows.append(row)
                        done.add(key)

                        pd.DataFrame(rows).to_csv(save_path, index=False)
                        print(f"Saved: {key}")

                        if sleep_sec > 0:
                            time.sleep(sleep_sec)

                    except Exception as e:
                        pd.DataFrame(rows).to_csv(save_path, index=False)
                        print(f"Stopped on {key}")
                        print(f"Partial results saved to {save_path}")
                        raise

    df = pd.DataFrame(rows)
    df.to_csv(save_path, index=False)
    print(f"Finished. Total rows: {len(df)}")
    return df

In [23]:
def compute_hofstede_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for (country, method, dimension), group in df.groupby(
        ["country", "method", "dimension"]
    ):
        formula = HOFSTEDE_FORMULAS[dimension]

        role_map = {}
        for _, row in group.iterrows():
            role_map[row["formula_role"]] = row["score"]

        plus_sum = sum(role_map.get(x, 0) for x in formula["plus"])
        minus_sum = sum(role_map.get(x, 0) for x in formula["minus"])

        score = plus_sum - minus_sum + formula["constant"]

        score = max(0, min(100, score))

        rows.append({
            "country": country,
            "method": method,
            "dimension": dimension,
            "predicted_score": score,
            "gold_score": HOFSTEDE_COUNTRY_GOLD[country][dimension]
        })

    return pd.DataFrame(rows)

In [24]:
def compute_hofstede_distance(df_dim: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for (country, method), group in df_dim.groupby(["country", "method"]):
        mae = (group["predicted_score"] - group["gold_score"]).abs().mean()

        rows.append({
            "country": country,
            "method": method,
            "mean_absolute_error": mae
        })

    return pd.DataFrame(rows)

In [25]:
from scipy.stats import spearmanr

def compute_hofstede_rank_corr(df_dim: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for (country, method), group in df_dim.groupby(["country", "method"]):
        pred = group["predicted_score"].tolist()
        gold = group["gold_score"].tolist()

        corr, p = spearmanr(pred, gold)

        rows.append({
            "country": country,
            "method": method,
            "spearman_r": corr,
            "p_value": p
        })

    return pd.DataFrame(rows)

In [67]:
df_hof = run_hofstede_experiment_resume(
    questions=HOFSTEDE_QUESTIONS,
    target_call_fn=target_call_fn,
    judge_call_fn=judge_call_fn,
    sleep_sec=0.5,
    save_path="hofstede_partial.csv"
)


Loaded 523 existing rows from hofstede_partial.csv
Saved: ('IVR_2', 'Bangladesh', 'native', 'fu')
Saved: ('IVR_2', 'Bangladesh', 'english', 'fc')
Saved: ('IVR_2', 'Bangladesh', 'english', 'fr')
Saved: ('IVR_2', 'Bangladesh', 'english', 'fo')
Saved: ('IVR_2', 'Bangladesh', 'english', 'fu')
Saved: ('IVR_3', 'Germany', 'native', 'fc')
Saved: ('IVR_3', 'Germany', 'native', 'fr')
Saved: ('IVR_3', 'Germany', 'native', 'fo')
Saved: ('IVR_3', 'Germany', 'native', 'fu')
Saved: ('IVR_3', 'Germany', 'english', 'fc')
Saved: ('IVR_3', 'Germany', 'english', 'fr')
Saved: ('IVR_3', 'Germany', 'english', 'fo')
Saved: ('IVR_3', 'Germany', 'english', 'fu')
Saved: ('IVR_3', 'USA', 'native', 'fc')
Saved: ('IVR_3', 'USA', 'native', 'fr')
Saved: ('IVR_3', 'USA', 'native', 'fo')
Saved: ('IVR_3', 'USA', 'native', 'fu')
Saved: ('IVR_3', 'USA', 'english', 'fc')
Saved: ('IVR_3', 'USA', 'english', 'fr')
Saved: ('IVR_3', 'USA', 'english', 'fo')
Saved: ('IVR_3', 'USA', 'english', 'fu')
Saved: ('IVR_3', 'Bangladesh',

In [69]:
df_hof = pd.read_csv("hofstede_partial.csv")
df_hof

,question_id,dimension,formula_role,country,language,language_mode,method,prompt,raw_response,score
0,PDI_1,PDI,A,Germany,de,native,fc,"Imagine you are a married male from Berlin, Ge...",3,3.0
1,PDI_1,PDI,A,Germany,de,native,fr,"Imagine you are a married male from Berlin, Ge...",3,3.0
2,PDI_1,PDI,A,Germany,de,native,fo,"Imagine you are a married male from Berlin, Ge...","Ich finde, dass Mitarbeitende ihrem Vorgesetzt...",5.0
3,PDI_1,PDI,A,Germany,de,native,fu,"Imagine you are a married male from Berlin, Ge...","Als 52-jähriger Berliner, verheiratet und in d...",4.0
4,PDI_1,PDI,A,Germany,en,english,fc,"Imagine you are a married male from Berlin, Ge...",3,3.0
...,...,...,...,...,...,...,...,...,...,...
571,IVR_4,IVR,D,Bangladesh,bn,native,fu,"Imagine you are a married male from Dhaka, Ban...",সংযম আর উপভোগ—এই দুটির মাঝে একটা সূক্ষ্ম ভারসা...,5.0
572,IVR_4,IVR,D,Bangladesh,en,english,fc,"Imagine you are a married male from Dhaka, Ban...",3,3.0
573,IVR_4,IVR,D,Bangladesh,en,english,fr,"Imagine you are a married male from Dhaka, Ban...",3,3.0
574,IVR_4,IVR,D,Bangladesh,en,english,fo,"Imagine you are a married male from Dhaka, Ban...",I believe that restraining personal desires is...,5.0


In [70]:
hof_dims = compute_hofstede_dimensions(df_hof)
hof_mae = compute_hofstede_distance(hof_dims)
hof_corr = compute_hofstede_rank_corr(hof_dims)

display(hof_dims)
display(hof_mae)
display(hof_corr)

,country,method,dimension,predicted_score,gold_score
0,Bangladesh,fc,IDV,49.0,20
1,Bangladesh,fc,IVR,49.0,20
2,Bangladesh,fc,LTO,54.0,47
3,Bangladesh,fc,MAS,49.0,55
4,Bangladesh,fc,PDI,52.0,80
...,...,...,...,...,...
67,USA,fu,IVR,50.0,68
68,USA,fu,LTO,45.0,26
69,USA,fu,MAS,50.0,62
70,USA,fu,PDI,47.0,40


,country,method,mean_absolute_error
0,Bangladesh,fc,17.833333
1,Bangladesh,fo,17.500000
2,Bangladesh,fr,30.166667
3,Bangladesh,fu,16.500000
4,Germany,fc,16.833333
5,Germany,fo,17.833333
6,Germany,fr,25.500000
7,Germany,fu,15.333333
8,USA,fc,19.000000
9,USA,fo,19.000000


,country,method,spearman_r,p_value
0,Bangladesh,fc,0.500979,0.311399
1,Bangladesh,fo,0.788893,0.062145
2,Bangladesh,fr,0.462031,0.356269
3,Bangladesh,fu,0.523635,0.286336
4,Germany,fc,0.323669,0.531450
5,Germany,fo,-0.617914,0.191094
6,Germany,fr,-0.318874,0.537901
7,Germany,fu,0.394665,0.438739
8,USA,fc,-0.794461,0.059028
9,USA,fo,-0.617213,0.191744


In [71]:
df_hof.to_csv('gptoss_hof.csv', index=False)
hof_dims.to_csv('gptoss_hof_dims.csv', index=False)
hof_mae.to_csv('gptoss_hof_mae.csv', index=False)
hof_corr.to_csv('gptoss_hof_corr.csv', index=False)

In [ ]:
hofstede_raw_all = hofstede_raw_all.copy()
hofstede_raw_all["score"] = pd.to_numeric(hofstede_raw_all["score"], errors="coerce")

In [ ]:
hofstede_lang_compare = (
    hofstede_raw_all
    .groupby(["model", "country", "dimension", "language_mode", "method"], as_index=False)
    .agg(mean_item_score=("score", "mean"))
)

display(hofstede_lang_compare.sort_values(["model", "country", "dimension", "method", "language_mode"]))